# HITL-IoT v1.1 — Leakage-Safe Track-1 Intrusion Detection Benchmark (Review Revision, Nine Models)

**Notebook:** `03_HITL-IoT_v1.1_Track1_IDS_Benchmark_REVISED.ipynb`

## 1. Purpose, Scope and Revision Continuity

This notebook extends the five-model Track-1 benchmark to **all nine model families evaluated in the reviewed manuscript**: Logistic Regression, SVM (RBF), Random Forest, XGBoost, LightGBM, MLP, 1D-CNN, LSTM and a Voting Ensemble. What changes relative to the original submission is the evaluation design, not the breadth of models:

| Aspect | Original submission | This revision |
|---|---|---|
| Data | HITL-IoT v1.0 | frozen HITL-IoT v1.1 (SHA-256 verified) |
| Features | broad diagnostic configuration | frozen `track1_primary_flow_only` policy |
| Split | 80/20 stratified random hold-out | chronological 70/15/15 (primary) + IID 70/15/15 (secondary) |
| Model selection | not separated from the test set | validation-only; test evaluated once per model |
| Probabilities | not released | raw held-out probabilities for all nine models |

**Research question.** How accurately can the nine reviewed model families distinguish benign from attack traffic using only the leakage-safe `track1_primary_flow_only` features under chronological evaluation?

> **Independence from benchmark-construction scores.**
> Every classifier in this notebook is trained exclusively on the frozen `track1_primary_flow_only` feature set. `generated_attack_score`, `generated_prediction`, `ambiguity_score`, labels, identifiers and oversight metadata are excluded from model inputs, and this is enforced by the programmatic leakage audit in Section 5. The resulting probabilities are independently learned classifier outputs, not reuse of variables embedded in the benchmark generator.

**Legacy neural baselines.** The 1D-CNN and LSTM baselines are retained to preserve continuity with the original reviewed experiment. They operate on the same leakage-safe encoded Track-1 representation and should be interpreted as comparative neural baselines rather than claims that convolutional or recurrent architectures are intrinsically optimal for independent flow-level tabular classification. Treating the encoded feature vector as an ordered one-dimensional sequence is a modelling convention inherited from the original experiment; it does not imply that the columns have a natural spatial or temporal order.

## How to run this notebook

1. Place this notebook in the **parent folder of `release_v1.1/`** (the frozen v1.1 release, unmodified).
2. Install the dependencies once (CPU builds are sufficient):
   `pip install numpy pandas scipy scikit-learn xgboost lightgbm tensorflow matplotlib joblib nbformat`
3. Use **Kernel → Restart & Run All**. Everything is written to `track1_benchmark_v1.1_review_revision/`.
4. **Expected cost.** Hyperparameter searches run only under the temporal protocol; IID reuses the selected configurations, SVM configurations are compared without Platt scaling, and inference is timed once. The LSTM (two protocols) remains the longest step; libsvm is single-threaded, so the SVM does not benefit from extra cores. Peak memory is about 3–4 GB.
   **Staged execution:** set `RUN_MODE = "train_only"` in Section 2 to train and checkpoint all models, then `RUN_MODE = "analysis_only"` to produce every table, figure and audit without retraining. `"full"` does both in one pass.
5. **Interrupted? Just run all cells again.** Long units also resume internally: SVM configurations are cached individually and neural networks resume from their last completed epoch. Completed units are reloaded from `checkpoints/` (after SHA-256 verification of their model files) and are not refitted; the run continues with the next unfinished unit. Delete the `checkpoints/` folder only if you deliberately want to retrain everything.
6. Progress lines are printed for every hyperparameter configuration and every neural-network epoch.

**Checkpointing.** Each (protocol, model) unit writes a checkpoint after its single test evaluation. If execution is interrupted, re-running the notebook reloads completed units instead of refitting them; a unit's test partition is never re-evaluated to change a result.

## 2. Imports and Reproducibility Configuration

One seed (42) is used for Python, NumPy, scikit-learn, XGBoost, LightGBM and TensorFlow; TensorFlow operator determinism is enabled where supported. Results are **reproducible under the documented software configuration and random seeds**; identical model fitting is not claimed across different library versions, hardware or thread counts.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import gc
import hashlib
import io
import itertools
import zipfile
import json
import platform
import random
import re
import sys
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils import Bunch
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    TF_DETERMINISM = True
except Exception:
    TF_DETERMINISM = False
N_JOBS = -1
THRESHOLD = 0.50
BOOTSTRAP_B, BOOTSTRAP_SEED = 1000, 2024
INFERENCE_WARMUP, INFERENCE_REPS = 2, 10
INFERENCE_MIN_REPS, INFERENCE_SLOW_CALL_S = 5, 15.0   # if one full-batch call exceeds 15 s, 5 timed repetitions are used
SINGLE_RECORD_N, SINGLE_RECORD_SEED = 500, 7
UNIT_VERSION = "rev2"      # bump to invalidate all checkpoints

# RUN_MODE controls staged execution:
#   "full"          train/resume all units, then run every analysis (default; one pass)
#   "train_only"    train/resume all 18 units and stop (no bootstrap, audits, figures)
#   "analysis_only" require all checkpoints, never train, and produce every table/figure/audit
RUN_MODE = "full"
assert RUN_MODE in ("full", "train_only", "analysis_only"), RUN_MODE

# Optional: reuse the temporal-validation-selected hyperparameters of the five core models from an earlier,
# hash-verified v1.1 five-model package (skips their searches; final models are still refitted here).
# Default False: a public release should regenerate every search from scratch.
FAST_RESUME_MODE = False
REFERENCE_CORE5_DIR = Path("track1_benchmark_v1.1")      # extracted earlier package
REFERENCE_CORE5_ZIP = Path("track1_benchmark_v1.1.zip")  # or its ZIP

RELEASE_DIR = Path("data/v1.1")
OUT_DIR = Path("track1_benchmark_v1.1_review_revision")
DIRS = {d: OUT_DIR / d for d in ("audit", "tables", "predictions", "probabilities", "models", "figures",
                                 "manuscript_values", "logs", "checkpoints")}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

VERSION = "v1.1"
FILES = {"dataset": RELEASE_DIR / f"HITL-IoT_dataset_{VERSION}.csv",
         "feature_policy": RELEASE_DIR / f"HITL-IoT_feature_policy_{VERSION}.json",
         "splits": RELEASE_DIR / f"HITL-IoT_splits_{VERSION}.csv",
         "schema": RELEASE_DIR / f"HITL-IoT_schema_{VERSION}.json",
         "manifest": RELEASE_DIR / f"HITL-IoT_manifest_{VERSION}.json",
         "checksums": RELEASE_DIR / f"HITL-IoT_checksums_{VERSION}.txt",
         "summary": RELEASE_DIR / f"HITL-IoT_summary_{VERSION}.json"}

MODEL_ORDER = ["Logistic Regression", "SVM (RBF)", "Random Forest", "XGBoost", "LightGBM", "MLP",
               "1D-CNN", "LSTM", "Voting Ensemble"]
MODEL_SLUG = {"Logistic Regression": "logistic_regression", "SVM (RBF)": "svm_rbf", "Random Forest": "random_forest",
              "XGBoost": "xgboost", "LightGBM": "lightgbm", "MLP": "mlp", "1D-CNN": "1d_cnn", "LSTM": "lstm",
              "Voting Ensemble": "voting_ensemble"}
MODEL_KIND = {"Logistic Regression": "sklearn", "SVM (RBF)": "svm", "Random Forest": "sklearn", "XGBoost": "sklearn",
              "LightGBM": "sklearn", "MLP": "sklearn", "1D-CNN": "keras", "LSTM": "keras", "Voting Ensemble": "voting"}
REPRESENTATIVE_TREES = ["Random Forest", "XGBoost", "LightGBM"]   # predeclared, not chosen from test results
ENSEMBLE_MEMBERS = ["XGBoost", "LightGBM", "Random Forest"]      # legacy composition (recovered)
PROTOCOLS = {"temporal": "temporal_split", "iid": "iid_split"}
TABLES_WRITTEN: list[Path] = []

ENVIRONMENT = {
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__, "xgboost": xgb.__version__, "lightgbm": lgb.__version__,
    "tensorflow": tf.__version__, "tf_op_determinism": TF_DETERMINISM, "scipy": scipy.__version__,
    "matplotlib": mpl.__version__, "joblib": joblib.__version__, "os": platform.platform(),
    "processor": platform.processor() or platform.machine(), "logical_cpus": os.cpu_count(),
    "n_jobs_setting": N_JOBS, "seed": SEED, "threshold": THRESHOLD,
    "bootstrap_resamples": BOOTSTRAP_B, "bootstrap_seed": BOOTSTRAP_SEED,
    "inference_warmup_calls": INFERENCE_WARMUP, "inference_timed_repetitions": INFERENCE_REPS,
    "inference_reduced_repetitions_if_call_exceeds_s": [INFERENCE_MIN_REPS, INFERENCE_SLOW_CALL_S], "run_mode": RUN_MODE,
    "reproducibility_statement": "reproducible under the documented software configuration and random seeds"}
(DIRS["logs"] / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2))
print(json.dumps(ENVIRONMENT, indent=2))

### 2.1 Helpers and publication figure style

The same style system as the v1.1 descriptive notebook. Nine models need more than colour to stay distinguishable, so each model has a fixed colour **and** a fixed line style **and** a fixed marker, used identically in every figure.

In [ ]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_table(df: pd.DataFrame, filename: str, subdir: str = "tables", float_format: str = "%.6g") -> Path:
    path = DIRS[subdir] / filename
    df.to_csv(path, index=False, float_format=float_format)
    TABLES_WRITTEN.append(path)
    return path


def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if np.isnan(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, (pd.Timestamp,)): return o.isoformat()
    if isinstance(o, (tuple, set)): return list(o)
    if isinstance(o, Path): return str(o)
    return str(o)


SINGLE_COL, DOUBLE_COL = 3.5, 7.16
MODEL_COLORS = {"Logistic Regression": "#0072B2", "SVM (RBF)": "#56B4E9", "Random Forest": "#009E73",
                "XGBoost": "#D55E00", "LightGBM": "#CC79A7", "MLP": "#E69F00", "1D-CNN": "#8C7A00",
                "LSTM": "#882255", "Voting Ensemble": "#1A1A1A"}
MODEL_MARKERS = {"Logistic Regression": "o", "SVM (RBF)": "P", "Random Forest": "s", "XGBoost": "^", "LightGBM": "D",
                 "MLP": "v", "1D-CNN": "X", "LSTM": "h", "Voting Ensemble": "*"}
MODEL_LINESTYLES = {"Logistic Regression": "-", "SVM (RBF)": (0, (1, 1)), "Random Forest": "--", "XGBoost": "-.",
                    "LightGBM": (0, (3, 1, 1, 1, 1, 1)), "MLP": (0, (5, 1)), "1D-CNN": (0, (3, 2)),
                    "LSTM": (0, (6, 2, 1, 2)), "Voting Ensemble": "-"}
MODEL_LW = {m: (1.6 if m == "Voting Ensemble" else 1.15) for m in MODEL_ORDER}
NEUTRAL, GRID = "#4D4D4D", "#E3E3E3"
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", ["#F4F7FA", "#B9CEE0", "#6A96BD", "#2B5C8A", "#15324F"])


def set_publication_style() -> None:
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "legend.fontsize": 7.5, "legend.frameon": False, "legend.handlelength": 2.2,
        "axes.linewidth": 0.6, "axes.edgecolor": "#333333", "axes.spines.top": False, "axes.spines.right": False,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5, "ytick.major.size": 2.5,
        "lines.linewidth": 1.2, "figure.dpi": 110, "savefig.dpi": 600,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", "axes.formatter.use_mathtext": False})


def ygrid(ax, axis="y"):
    ax.grid(True, axis=axis, color=GRID, linewidth=0.5); ax.set_axisbelow(True)


def add_panel_label(ax, letter, x=-0.16, y=1.03):
    ax.text(x, y, f"({letter})", transform=ax.transAxes, fontsize=10.5, fontweight="bold", va="bottom", ha="left")


FIGURE_REGISTRY: dict[str, dict] = {}


def save_publication_figure(fig, stem, caption, n_panels, n_models):
    paths = {}
    for ext, kw in (("pdf", {}), ("svg", {}), ("png", {"dpi": 600})):
        p = DIRS["figures"] / f"{stem}.{ext}"
        fig.savefig(p, bbox_inches="tight", pad_inches=0.03, **kw)
        paths[ext] = p
    FIGURE_REGISTRY[stem] = {"paths": paths, "caption": caption, "size_in": tuple(np.round(fig.get_size_inches(), 2)),
                             "n_panels": n_panels, "n_models": n_models,
                             "texts": " ".join(t.get_text() for ax in fig.axes for t in
                                               [ax.xaxis.label, ax.yaxis.label, *ax.texts, *ax.get_xticklabels(), *ax.get_yticklabels()])}
    return paths


set_publication_style()
print("Font:", mpl.font_manager.FontProperties(family=mpl.rcParams["font.sans-serif"]).get_name(),
      "| TF op determinism:", TF_DETERMINISM)

## 3. Frozen Release Verification

Every release artefact is hashed and compared with `HITL-IoT_checksums_v1.1.txt`; the notebook stops on any mismatch.

In [ ]:
missing = [str(p) for k, p in FILES.items() if k != "summary" and not p.is_file()]
assert not missing, f"Required release files missing: {missing}"
expected = {}
for line in FILES["checksums"].read_text(encoding="utf-8").splitlines():
    if line.strip():
        d, n = line.strip().split(maxsplit=1); expected[n.strip()] = d.lower()
rows = []
for key in ("dataset", "feature_policy", "splits", "schema", "manifest", "summary"):
    p = FILES[key]
    if p.is_file():
        obs = sha256_of(p)
        rows.append({"File": p.name, "Expected SHA-256": expected.get(p.name), "Observed SHA-256": obs,
                     "Match": expected.get(p.name) == obs})
source_verification = pd.DataFrame(rows)
save_table(source_verification, "source_verification.csv", "audit")
assert source_verification["Match"].all(), "Frozen release verification failed — refusing to proceed."
RELEASE_VERIFIED = True
manifest = json.loads(FILES["manifest"].read_text(encoding="utf-8"))
print(f"[PASS] Frozen v1.1 source verified ({manifest['dataset_name']} {manifest['version']}, seed {manifest['seed']})")
_obs = source_verification.set_index("File")["Observed SHA-256"]
BENCHMARK_CONTEXT = {"dataset_sha256": _obs[FILES["dataset"].name], "feature_policy_sha256": _obs[FILES["feature_policy"].name],
                     "splits_sha256": _obs[FILES["splits"].name], "schema_sha256": _obs[FILES["schema"].name],
                     "seed": SEED, "threshold": THRESHOLD, "feature_policy_key": "track1_primary_flow_only",
                     "protocol": "temporal-selection, fixed-configuration IID", "unit_version": UNIT_VERSION}
CONTEXT_FINGERPRINT = hashlib.sha256(json.dumps(BENCHMARK_CONTEXT, sort_keys=True).encode()).hexdigest()
(DIRS["logs"] / "benchmark_context.json").write_text(json.dumps({**BENCHMARK_CONTEXT, "context_fingerprint": CONTEXT_FINGERPRINT}, indent=2))
print("Benchmark context fingerprint:", CONTEXT_FINGERPRINT)

## 4. Feature Policy

The primary feature list is read from `track1_primary_flow_only`; it is not typed, extended or pruned by hand.

In [ ]:
policy = json.loads(FILES["feature_policy"].read_text(encoding="utf-8"))
schema = pd.DataFrame(json.loads(FILES["schema"].read_text(encoding="utf-8"))).set_index("field")
FEATURES = list(policy["track1_primary_flow_only"])
df = pd.read_csv(FILES["dataset"], low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
assert set(FEATURES) <= set(df.columns)

def feature_kind(s):
    if pd.api.types.is_bool_dtype(s): return "boolean (cast to 0/1)"
    if pd.api.types.is_numeric_dtype(s): return "numeric"
    return "categorical"

feature_audit = pd.DataFrame([{"feature": f, "dtype": str(df[f].dtype), "kind": feature_kind(df[f]),
                               "missing": int(df[f].isna().sum()), "unique_values": int(df[f].nunique()),
                               "schema_category": schema.at[f, "category"] if f in schema.index else None} for f in FEATURES])
save_table(feature_audit, "track1_feature_audit.csv", "audit")
CATEGORICAL = feature_audit.loc[feature_audit.kind == "categorical", "feature"].tolist()
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
print(f"{len(FEATURES)} primary Track-1 features ({len(NUMERIC)} numeric/boolean, {len(CATEGORICAL)} categorical: {CATEGORICAL})")

## 5. Track-1 Leakage Audit

Assertions stop execution if any label, construction signal, oversight variable or identifier reaches the primary feature set.

In [ ]:
LABELS = {"is_attack", "attack_type"}
CONSTRUCTION = {"generated_attack_score", "generated_prediction", "ambiguity_score", "configured_intrinsic_risk", "risk_score"}
OVERSIGHT = {"oversight_selected", "oversight_profile", "profile_decision", "profile_confidence",
             "generated_decision_latency", "model_profile_agreement", "profile_rationale"}
IDENTIFIERS = {"flow_id", "timestamp", "src_ip", "dst_ip", "src_mac", "dst_mac", "device_name", "src_port"}
schema_prohibited = set(schema.index[schema["category"].isin(
    ["identifier", "target_label", "benchmark_construction_variable", "generated_oversight_metadata"])])
F = set(FEATURES)
leak_checks = [("No label leakage", not (F & LABELS)), ("No construction-signal leakage", not (F & CONSTRUCTION)),
               ("No oversight-variable leakage", not (F & OVERSIGHT)), ("No identifier leakage", not (F & IDENTIFIERS)),
               ("No schema-prohibited variable", not (F & schema_prohibited)),
               ("Contextual dst_category absent from primary experiment", "dst_category" not in F),
               ("Primary feature list matches feature-policy JSON", FEATURES == policy["track1_primary_flow_only"])]
for n, ok in leak_checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {n}")
assert all(ok for _, ok in leak_checks), "Leakage audit failed."
LEAKAGE_AUDIT_PASS = True
save_table(pd.DataFrame(leak_checks, columns=["check", "passed"]), "track1_leakage_audit.csv", "audit")

## 6. Official Splits

The released split file is merged on `flow_id`; both protocols must cover each flow once with ~70/15/15 proportions, and the temporal partitions must be strictly chronological.

In [ ]:
splits = pd.read_csv(FILES["splits"])
assert splits["flow_id"].is_unique and set(splits.flow_id) == set(df.flow_id)
df = df.merge(splits[["flow_id", "temporal_split", "iid_split"]], on="flow_id", how="left", validate="one_to_one")
PARTS = ["train", "validation", "test"]; TARGET = {"train": 0.70, "validation": 0.15, "test": 0.15}
rows = []
for proto, col in PROTOCOLS.items():
    for part in PARTS:
        m = df[col] == part
        rows.append({"protocol": proto, "partition": part, "flows": int(m.sum()), "proportion": m.mean(),
                     "attack_count": int(df.loc[m, "is_attack"].sum()), "attack_rate": df.loc[m, "is_attack"].mean(),
                     "first_timestamp": df.loc[m, "timestamp"].min(), "last_timestamp": df.loc[m, "timestamp"].max()})
split_integrity = pd.DataFrame(rows)
t = split_integrity[split_integrity.protocol == "temporal"].set_index("partition")
split_checks = {
    "every flow in one temporal partition": df["temporal_split"].isin(PARTS).all(),
    "every flow in one IID partition": df["iid_split"].isin(PARTS).all(),
    "temporal ~70/15/15": all(abs(t.at[p, "proportion"] - TARGET[p]) <= 0.005 for p in PARTS),
    "IID ~70/15/15": all(abs(r.proportion - TARGET[r.partition]) <= 0.005 for r in split_integrity[split_integrity.protocol == "iid"].itertuples()),
    "max(train ts) < min(validation ts)": t.at["train", "last_timestamp"] < t.at["validation", "first_timestamp"],
    "max(validation ts) < min(test ts)": t.at["validation", "last_timestamp"] < t.at["test", "first_timestamp"]}
for k, v in split_checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
assert all(split_checks.values())
TEMPORAL_SPLIT_OK = IID_SPLIT_OK = True
save_table(split_integrity, "split_integrity.csv", "audit")

y_all = df["is_attack"].astype(int).to_numpy()
X_all = df[FEATURES].copy()
for c in X_all.columns:
    if pd.api.types.is_bool_dtype(X_all[c]):
        X_all[c] = X_all[c].astype(np.int8)

def partition(proto, part):
    m = (df[PROTOCOLS[proto]] == part).to_numpy()
    return X_all.loc[m], y_all[m], df.loc[m, ["flow_id", "timestamp"]].reset_index(drop=True)

split_integrity.style.format({"proportion": "{:.4f}", "attack_rate": "{:.4f}"}).hide(axis="index")

## 7. Preprocessing, Metrics and Timing

* All preprocessing is fitted on the training partition only. Numeric/boolean: median imputation, plus `StandardScaler` for Logistic Regression, SVM, MLP, 1D-CNN and LSTM. Categorical: most-frequent imputation and `OneHotEncoder(handle_unknown="ignore")`.
* Positive class = attack; threshold 0.50. FPR = FP/(FP+TN); FNR = FN/(FN+TP).
* **Training time** denotes fitting time of the selected final configuration and excludes hyperparameter-search overhead.
* **Amortized batch inference time.** Probability inference was measured end-to-end on the full held-out partition after warm-up. Median total execution time across repeated runs was divided by the number of test samples; consequently, the reported quantity is amortized batch inference time per sample rather than single-flow request latency.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    num = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale_numeric else [])
    cat = [("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
    return ColumnTransformer([("num", Pipeline(num), NUMERIC), ("cat", Pipeline(cat), CATEGORICAL)],
                             remainder="drop", verbose_feature_names_out=False)

_Xtr, _, _ = partition("temporal", "train")
_demo = make_preprocessor(True).fit(_Xtr)
assert np.allclose(_demo.named_transformers_["num"].named_steps["scale"].mean_, _Xtr[NUMERIC].mean().to_numpy())
assert not np.allclose(_demo.named_transformers_["num"].named_steps["scale"].mean_, X_all[NUMERIC].mean().to_numpy())
N_ENCODED = _demo.transform(_Xtr.iloc[:5]).shape[1]
print(f"[PASS] preprocessors fitted on training data only ({len(FEATURES)} features -> {N_ENCODED} encoded inputs)")
del _demo, _Xtr


def safe_div(a, b):
    return a / b if b else float("nan")


def compute_metrics(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(np.unique(y)) == 2
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred) if both else float("nan"),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0), "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "mcc": matthews_corrcoef(y, pred) if both else float("nan"),
            "auroc": roc_auc_score(y, p) if both else float("nan"),
            "fpr": safe_div(fp, fp + tn), "fnr": safe_div(fn, fn + tp), "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}


def time_inference(model, X):
    for _ in range(INFERENCE_WARMUP):
        model.predict_proba(X)
    runs = []
    t0 = time.perf_counter(); model.predict_proba(X); runs.append(time.perf_counter() - t0)
    reps = INFERENCE_MIN_REPS if runs[0] > INFERENCE_SLOW_CALL_S else INFERENCE_REPS
    for _ in range(reps - 1):
        t0 = time.perf_counter(); model.predict_proba(X); runs.append(time.perf_counter() - t0)
    med = float(np.median(runs))
    return {"batch_inference_total_s": med, "amortized_batch_inference_ms_per_sample": 1000 * med / len(X),
            "batch_inference_samples_per_s": len(X) / med, "timed_repetitions": len(runs)}

## 8. Model Configurations and Selection Rule

**Selection rule (predeclared):** highest validation macro-F1; ties broken by MCC, AUROC, balanced accuracy, then declared configuration order. Test partitions play no role in selection. Selected configurations are not refitted on train + validation, so validation probabilities stay out-of-sample for Step 7.

**Continuity of configurations.**

* *Logistic Regression, Random Forest, XGBoost, LightGBM, MLP:* the search spaces of the previous five-model package are kept unchanged so that Section 10 is a true regression check. The legacy single configurations (e.g. Random Forest `n_estimators=100, max_depth=20`) are documented in the continuity audit.
* *SVM (RBF):* four configurations on the full training partition (no subsampling): `gamma="scale"` (the legacy, feature-scale-aware setting), `C ∈ {1, 10}`, `class_weight ∈ {None, "balanced"}`; the legacy configuration is candidate 0. Because Platt scaling (`probability=True`) multiplies SVM fitting cost, configurations are compared using the SVM **decision function** (sign rule for predictions, decision values for AUROC), and only the selected configuration is refitted with `probability=True`. All reported SVM metrics and saved probabilities come from that final probabilistic model.
* *IID protocol:* every model reuses the configuration selected on temporal validation (Section 15).
* *1D-CNN and LSTM:* the **exact legacy architectures** recovered from the original baseline notebook (Conv1D 64–128–64 with max-pooling, dropout and global average pooling; stacked LSTM 128–64 with dropout), with the legacy training settings (Adam 1e-3, binary cross-entropy, batch 256, ≤ 50 epochs, early stopping patience 10 with best-weight restoration, learning-rate reduction ×0.5 after 5 stalled epochs). The only change is that early stopping monitors the **official validation partition** instead of a random 20% slice of training data. No architecture search is performed.
* *Stabilised legacy LSTM.* In a pilot run of this notebook, the exact legacy LSTM (Adam without clipping, raw standardised inputs) failed to learn under the v1.1 Track-1 representation: its validation loss stayed at the prevalence baseline (≈0.42, the log-loss of always predicting the attack rate) and its training loss increased after epoch 4, indicating optimisation instability rather than overfitting. Diagnosis on temporal training data showed standardised inputs up to |z| ≈ 190 from heavy-tailed byte features (0.09% of inputs exceed |z| = 5), which saturate recurrent gates; the 1D-CNN learned normally on the same inputs. Following a documented, reproducible correction, the LSTM keeps the legacy architecture and training settings but adds **gradient-norm clipping (clipnorm = 1.0)** and **capping of standardised inputs at ±5 SD**. The decision used temporal train/validation evidence only; the unstabilised learning curve is kept in `audit/legacy_lstm_unstabilised_learning_curve.json`. The capping is applied to the LSTM only, so the 1D-CNN remains the unmodified legacy configuration.
* *LSTM input representation.* The stabilised LSTM with the legacy sequence framing (75 time steps of one scalar each) also remained at the prevalence baseline for 11 epochs on the full training data (`audit/stabilised_lstm_learning_curve.json`), while the 1D-CNN learned normally from the identical framing. The reported LSTM therefore presents each flow's encoded features as **one time step with 75 values** (input shape 1 × 75); the legacy LSTM layers, training settings, gradient clipping and ±5 SD input capping are unchanged. With a single time step the recurrent layers act as a gated feed-forward transformation; the model is reported as a continuity baseline and makes no claim about sequential structure. This decision used temporal train/validation evidence only.
* *Voting Ensemble:* the **legacy composition** recovered from the original notebook — equal-weight soft voting over XGBoost, LightGBM and Random Forest — assembled from the already-selected Track-1 members of the same protocol. No weights are tuned.
* *MLP:* the current scikit-learn MLP (with its validation grid) is retained so the five-model regression check remains exact; the legacy Keras MLP (256–128–64–32) is recorded in the continuity audit.

In [ ]:
# Stabilised legacy LSTM (see Section 8): same architecture and training settings, plus gradient-norm clipping and
# capping of standardised inputs at +/-5 SD. Decided from temporal train/validation evidence only.
LSTM_STABILISATION = {"clipnorm": 1.0, "input_clip_sd": 5.0}
SVM_CACHE_MB = 2048   # kernel cache in MB; raise to 4096 if RAM allows (affects speed only, not results)
KERAS_SETTINGS = {"epochs": 50, "batch_size": 256, "patience": 10, "reduce_lr_factor": 0.5, "reduce_lr_patience": 5,
                  "min_lr": 1e-6, "learning_rate": 1e-3, "loss": "binary_crossentropy", "optimizer": "adam"}
SEARCH_SPACES = {
    "Logistic Regression": [{"C": C, "class_weight": cw} for C in (0.1, 1.0, 10.0) for cw in (None, "balanced")],
    "SVM (RBF)": [{"C": C, "gamma": "scale", "class_weight": cw} for C in (1.0, 10.0) for cw in (None, "balanced")],
    "Random Forest": [{"n_estimators": 300, "max_depth": d, "min_samples_leaf": l, "class_weight": cw}
                      for d in (None, 20) for l in (1, 2) for cw in (None, "balanced_subsample")],
    "XGBoost": [{"n_estimators": 400, "max_depth": d, "learning_rate": lr, "subsample": 0.8, "colsample_bytree": 0.8}
                for d in (4, 8) for lr in (0.05, 0.1)],
    "LightGBM": [{"n_estimators": 400, "num_leaves": nl, "max_depth": -1, "learning_rate": lr, "colsample_bytree": 0.8}
                 for nl in (31, 127) for lr in (0.05, 0.1)],
    "MLP": [{"hidden_layer_sizes": h, "alpha": a, "learning_rate_init": 1e-3} for h in ((128, 64), (256, 128, 64)) for a in (1e-4, 1e-3)],
    "1D-CNN": [{"architecture": "legacy Conv1D(64,3)-MaxPool(2)-Dropout(0.3)-Conv1D(128,3)-MaxPool(2)-Dropout(0.3)-Conv1D(64,3)-GAP-Dense(64)-Dropout(0.3)-Dense(1,sigmoid)", **KERAS_SETTINGS}],
    "LSTM": [{"architecture": "legacy LSTM(128,seq)-Dropout(0.3)-LSTM(64)-Dropout(0.3)-Dense(32)-Dropout(0.2)-Dense(1,sigmoid)", **KERAS_SETTINGS,
              "stabilisation": {"gradient_clipnorm": LSTM_STABILISATION["clipnorm"], "input_clip_sd": LSTM_STABILISATION["input_clip_sd"]},
              "input_representation": "single time step: (1, n_encoded_features)"}],
    "Voting Ensemble": [{"members": ENSEMBLE_MEMBERS, "voting": "soft", "weights": "equal"}],
}
LEGACY_CONFIGS = {
    "Logistic Regression": "LogisticRegression(max_iter=1000), scaled inputs",
    "SVM (RBF)": "SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)",
    "Random Forest": "n_estimators=100, max_depth=20, min_samples_split=5, min_samples_leaf=2 (manuscript: n_estimators=100, max_depth=20)",
    "XGBoost": "notebook: n_estimators=100, max_depth=10, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8; manuscript text: max_depth=6 (inconsistent)",
    "LightGBM": "notebook: n_estimators=100, max_depth=10, learning_rate=0.1, num_leaves=31; manuscript text: learning_rate=0.05, n_estimators=150 (inconsistent)",
    "MLP": "Keras Dense 256-128-64-32 with dropout, Adam, validation_split=0.2",
    "1D-CNN": "Keras Conv1D 64-128-64 (as retained here), validation_split=0.2",
    "LSTM": "Keras LSTM 128-64 layers retained, validation_split=0.2, input (n_features, 1); revised run: gradient clipping (clipnorm=1.0), +/-5 SD input capping, single-time-step input (1, n_features)",
    "Voting Ensemble": "soft VotingClassifier over XGBoost, LightGBM, Random Forest (as retained here)"}


CORE5 = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "MLP"]
VERIFIED_CORE5_PARAMS: dict = {}


def _read_reference(rel):
    p = REFERENCE_CORE5_DIR / rel
    if p.is_file():
        return p.read_bytes()
    if REFERENCE_CORE5_ZIP.is_file():
        with zipfile.ZipFile(REFERENCE_CORE5_ZIP) as zf:
            for name in (rel, f"track1_benchmark_v1.1/{rel}"):
                if name in zf.namelist():
                    return zf.read(name)
    return None


def import_verified_core5_params() -> bool:
    if not FAST_RESUME_MODE:
        return False
    hp_b, sv_b, log_b = (_read_reference(r) for r in ("tables/selected_hyperparameters.csv", "audit/source_verification.csv",
                                                        "tables/hyperparameter_search_temporal.csv"))
    if hp_b is None or sv_b is None or log_b is None:
        print("[INFO] FAST_RESUME_MODE: reference package incomplete; full temporal searches will run."); return False
    sv = pd.read_csv(io.BytesIO(sv_b)).set_index("File")
    for key in ("dataset", "feature_policy", "splits"):   # reference must have been produced from THIS frozen release
        if sv.at[FILES[key].name, "Observed SHA-256"].lower() != source_verification.set_index("File").at[FILES[key].name, "Observed SHA-256"]:
            print(f"[INFO] FAST_RESUME_MODE: reference hash mismatch for {FILES[key].name}; full searches will run."); return False
    hp = pd.read_csv(io.BytesIO(hp_b)); hp = hp[hp["protocol"] == "temporal"]
    for m in CORE5:
        row = hp[hp["model"] == m]
        if row.empty:
            return False
        VERIFIED_CORE5_PARAMS[m] = json.loads(row.iloc[0]["hyperparameters"])
    # provenance: copy the earlier search log (the evidence behind the reused choices) into this package
    (DIRS["audit"] / "reused_core5_temporal_search_log.csv").write_bytes(log_b)
    (DIRS["audit"] / "reused_core5_selected_hyperparameters.csv").write_bytes(hp_b)
    print("[PASS] FAST_RESUME_MODE: verified temporal-selected hyperparameters imported for the five core models "
          "(search log copied to audit/reused_core5_temporal_search_log.csv); final models are refitted here.")
    return True


HAVE_VERIFIED_CORE5_PARAMS = import_verified_core5_params()


def build_pipeline(model, params):
    if model == "Logistic Regression":
        return Pipeline([("pre", make_preprocessor(True)), ("clf", LogisticRegression(solver="lbfgs", max_iter=3000, random_state=SEED, **params))])
    if model == "SVM (RBF)":
        return Pipeline([("pre", make_preprocessor(True)), ("clf", SVC(kernel="rbf", probability=True, cache_size=SVM_CACHE_MB, random_state=SEED, **params))])
    if model == "Random Forest":
        return Pipeline([("pre", make_preprocessor(False)), ("clf", RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS, **params))])
    if model == "XGBoost":
        return Pipeline([("pre", make_preprocessor(False)), ("clf", xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                          tree_method="hist", random_state=SEED, n_jobs=N_JOBS, **params))])
    if model == "LightGBM":
        return Pipeline([("pre", make_preprocessor(False)), ("clf", lgb.LGBMClassifier(objective="binary", random_state=SEED, n_jobs=N_JOBS,
                          deterministic=True, force_row_wise=True, verbose=-1, **params))])
    if model == "MLP":
        return Pipeline([("pre", make_preprocessor(True)), ("clf", MLPClassifier(activation="relu", solver="adam", batch_size=256, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10, random_state=SEED, **params))])
    raise ValueError(model)


def build_keras(model, n_inputs):
    L = tf.keras.layers
    body = ([L.Conv1D(64, 3, activation="relu", padding="same"), L.MaxPooling1D(2), L.Dropout(0.3),
             L.Conv1D(128, 3, activation="relu", padding="same"), L.MaxPooling1D(2), L.Dropout(0.3),
             L.Conv1D(64, 3, activation="relu", padding="same"), L.GlobalAveragePooling1D(),
             L.Dense(64, activation="relu"), L.Dropout(0.3), L.Dense(1, activation="sigmoid")] if model == "1D-CNN" else
            [L.LSTM(128, return_sequences=True), L.Dropout(0.3), L.LSTM(64), L.Dropout(0.3),
             L.Dense(32, activation="relu"), L.Dropout(0.2), L.Dense(1, activation="sigmoid")])
    shape = (1, n_inputs) if model == "LSTM" else (n_inputs, 1)   # LSTM: one time step carrying all encoded features
    net = tf.keras.Sequential([L.Input(shape=shape)] + body, name=MODEL_SLUG[model])
    opt = (tf.keras.optimizers.Adam(KERAS_SETTINGS["learning_rate"], clipnorm=LSTM_STABILISATION["clipnorm"]) if model == "LSTM"
           else tf.keras.optimizers.Adam(KERAS_SETTINGS["learning_rate"]))
    net.compile(optimizer=opt, loss="binary_crossentropy")
    return net


class KerasProbabilityModel:
    # End-to-end wrapper (train-fitted preprocessor + Keras network) exposing predict_proba for timing and evaluation.
    def __init__(self, pre, net, clip=None, single_step=False):
        self.pre, self.net, self.clip, self.single_step = pre, net, clip, single_step
    def transform(self, X):
        A = self.pre.transform(X)
        if self.clip is not None:
            A = np.clip(A, -self.clip, self.clip)
        A = A.astype("float32")
        return A[:, None, :] if self.single_step else A[..., None]
    def predict_proba(self, X):
        p = self.net.predict(self.transform(X), batch_size=4096, verbose=0).ravel().astype(float)
        p = np.clip(p, 0.0, 1.0)
        return np.column_stack([1 - p, p])
    def predict_one(self, X):
        p = float(self.net(self.transform(X), training=False).numpy().ravel()[0])
        return p


def assemble_voting(members: dict) -> VotingClassifier:
    # Soft VotingClassifier built from already-fitted member pipelines (no refitting, equal weights).
    vc = VotingClassifier(estimators=[(MODEL_SLUG[m], members[m]) for m in ENSEMBLE_MEMBERS], voting="soft")
    vc.estimators_ = [members[m] for m in ENSEMBLE_MEMBERS]
    vc.named_estimators_ = Bunch(**{MODEL_SLUG[m]: members[m] for m in ENSEMBLE_MEMBERS})
    vc.le_ = LabelEncoder().fit([0, 1]); vc.classes_ = vc.le_.classes_
    return vc

## 9. Checkpointed Training Units

One unit = one (protocol, model) pair: validation search → configuration frozen → single test evaluation → timing → serialisation → checkpoint. The selection key uses validation metrics only; the test partition is touched only after the configuration is fixed.

In [ ]:
RESULTS: dict[tuple, dict] = {}


def space_for(proto, model):
    # Temporal: the predeclared search space. IID: exactly the configuration selected on temporal validation.
    if proto == "temporal" and model in VERIFIED_CORE5_PARAMS:
        return [VERIFIED_CORE5_PARAMS[model]]
    if proto == "temporal" or MODEL_KIND[model] in ("keras", "voting"):
        return SEARCH_SPACES[model]
    assert ("temporal", model) in RESULTS, f"temporal unit for {model} must be completed before IID"
    return [RESULTS[("temporal", model)]["params"]]
TEST_EVALUATIONS: dict[tuple, int] = {}
SELECTION_KEYS = ["macro_f1", "mcc", "auroc", "balanced_accuracy"]


def model_paths(proto, model):
    slug = f"{proto}_{MODEL_SLUG[model]}"
    if MODEL_KIND[model] == "keras":
        return {"model": DIRS["models"] / f"{slug}.keras", "preprocessor": DIRS["models"] / f"{slug}_preprocessor.joblib"}
    return {"model": DIRS["models"] / f"{slug}.joblib"}


def finalize(proto, model, sel, fitted, log):
    # Single test evaluation after configuration selection, timing and serialisation.
    Xva, yva, _ = partition(proto, "validation")
    Xte, yte, _ = partition(proto, "test")
    p_te = fitted.predict_proba(Xte)[:, 1]
    TEST_EVALUATIONS[(proto, model)] = TEST_EVALUATIONS.get((proto, model), 0) + 1
    assert TEST_EVALUATIONS[(proto, model)] == 1, f"test evaluated twice for {(proto, model)}"
    timing = (time_inference(fitted, Xte) if proto == "temporal" else
              {"batch_inference_total_s": np.nan, "amortized_batch_inference_ms_per_sample": np.nan,
               "batch_inference_samples_per_s": np.nan, "timed_repetitions": 0})   # timing measured once, under the primary protocol
    paths = model_paths(proto, model)
    if MODEL_KIND[model] == "keras":
        fitted.net.save(paths["model"]); joblib.dump(fitted.pre, paths["preprocessor"], compress=3)
    else:
        joblib.dump(fitted, paths["model"], compress=3)
    files = {k: {"file": p.name, "sha256": sha256_of(p), "bytes": p.stat().st_size} for k, p in paths.items()}
    res = {"unit_version": UNIT_VERSION, "protocol": proto, "model": model, "search_space": space_for(proto, model),
           "hyperparameter_source": ("reused from earlier hash-verified temporal selection (FAST_RESUME_MODE)" if proto == "temporal" and model in VERIFIED_CORE5_PARAMS
                                     else "temporal validation search" if proto == "temporal" else "reused from temporal selection"),
           "search_log": log, "params": sel["params"], "config_index": sel["config_index"],
           "val_metrics": sel["val_metrics"], "p_val": sel["p_val"].astype(float), "p_test": p_te.astype(float),
           "test_metrics": compute_metrics(yte, p_te), "fit_time_s": sel["fit_time_s"], "timing": timing,
           "files": files, "n_train": int((df[PROTOCOLS[proto]] == "train").sum()), "n_val": len(yva), "n_test": len(yte),
           "extra": sel.get("extra", {})}
    meta = {k: res[k] for k in ("model", "protocol", "params", "config_index", "fit_time_s", "n_train", "n_val", "n_test", "files", "extra")}
    meta.update({"feature_policy": "track1_primary_flow_only", "features": FEATURES, "random_seed": SEED,
                 "training_partition": f"{proto} train", "threshold": THRESHOLD, "legacy_configuration": LEGACY_CONFIGS[model],
                 "package_versions": ENVIRONMENT, "trained_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                 "probability_files": [f"{proto}_validation_probabilities_all9.csv", f"{proto}_test_probabilities_all9.csv"]})
    (DIRS["models"] / f"{proto}_{MODEL_SLUG[model]}_metadata.json").write_text(json.dumps(meta, indent=2, default=jsonable))
    return res


def search_sklearn(proto, model):
    Xtr, ytr, _ = partition(proto, "train"); Xva, yva, _ = partition(proto, "validation")
    best, log = None, []
    for i, params in enumerate(space_for(proto, model)):
        pipe = build_pipeline(model, params)
        t0 = time.perf_counter(); pipe.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
        p_va = pipe.predict_proba(Xva)[:, 1]; m = compute_metrics(yva, p_va)
        log.append({"config_index": i, "params": json.dumps(params, default=jsonable), "fit_time_s": fit_s, **{f"val_{k}": v for k, v in m.items()}})
        key = tuple(m[k] for k in SELECTION_KEYS) + (-i,)
        print(f"    [{proto}] {model} config {i}: val macro-F1 {m['macro_f1']:.4f} ({fit_s:.1f}s)", flush=True)
        if best is None or key > best["key"]:
            best = {"key": key, "params": params, "config_index": i, "val_metrics": m, "p_val": p_va, "fit_time_s": fit_s, "fitted": pipe}
        else:
            del pipe
            gc.collect()
    return best, log


def search_svm(proto, model):
    # Stage 1 (temporal only): compare configurations with the SVM decision function (no Platt scaling), selecting on
    # validation macro-F1 -> MCC -> AUROC -> balanced accuracy; predictions use the sign of the decision function and
    # AUROC uses decision values. Stage 2: refit ONLY the selected configuration with probability=True.
    Xtr, ytr, _ = partition(proto, "train"); Xva, yva, _ = partition(proto, "validation")
    space = space_for(proto, model); log = []
    if len(space) > 1:
        best = None
        stage1 = DIRS["checkpoints"] / f"{proto}_svm_stage1_cache.joblib"      # per-configuration resume cache
        cache = joblib.load(stage1) if stage1.is_file() else {}
        for i, params in enumerate(space):
            ckey = (UNIT_VERSION, json.dumps(params, default=jsonable))
            if ckey in cache:
                fit_s, m = cache[ckey]
                print(f"    [{proto}] SVM (RBF) config {i}: reloaded from stage-1 cache", flush=True)
            else:
                pipe = build_pipeline(model, params); pipe.set_params(clf__probability=False)
                t0 = time.perf_counter(); pipe.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
                m = compute_metrics(yva, pipe.decision_function(Xva), threshold=0.0)
                del pipe
                cache[ckey] = (fit_s, m); joblib.dump(cache, stage1)
            log.append({"config_index": i, "params": json.dumps(params, default=jsonable), "fit_time_s": fit_s,
                        "selection_score": "decision_function (sign rule)", **{f"val_{k}": v for k, v in m.items()}})
            key = tuple(m[k] for k in SELECTION_KEYS) + (-i,)
            print(f"    [{proto}] SVM (RBF) config {i} (decision function): val macro-F1 {m['macro_f1']:.4f} ({fit_s:.0f}s)", flush=True)
            if best is None or key > best[0]:
                best = (key, i, params)
        sel_i, sel_params = best[1], best[2]
    else:
        sel_i, sel_params = 0, space[0]
    pipe = build_pipeline(model, sel_params)
    t0 = time.perf_counter(); pipe.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
    p_va = pipe.predict_proba(Xva)[:, 1]; m = compute_metrics(yva, p_va)
    if len(space) == 1:
        log.append({"config_index": 0, "params": json.dumps(sel_params, default=jsonable), "fit_time_s": fit_s,
                    "selection_score": "single configuration", **{f"val_{k}": v for k, v in m.items()}})
    print(f"    [{proto}] SVM (RBF) final probability fit (config {sel_i}): val macro-F1 {m['macro_f1']:.4f} ({fit_s:.0f}s)", flush=True)
    return {"params": sel_params, "config_index": sel_i, "val_metrics": m, "p_val": p_va, "fit_time_s": fit_s, "fitted": pipe,
            "extra": {"selection_procedure": "decision-function comparison, then single Platt-probability refit",
                      "final_fit_includes_probability_calibration": True}}, log


def search_keras(proto, model):
    Xtr, ytr, _ = partition(proto, "train"); Xva, yva, _ = partition(proto, "validation")
    pre = make_preprocessor(True).fit(Xtr)
    clip = LSTM_STABILISATION["input_clip_sd"] if model == "LSTM" else None
    _wrap = KerasProbabilityModel(pre, None, clip, single_step=(model == "LSTM"))
    A_tr = _wrap.transform(Xtr); A_va = _wrap.transform(Xva)
    print(f"    [{proto}] {model} input tensor: train {A_tr.shape}, validation {A_va.shape}", flush=True)
    expected = (1, N_ENCODED) if model == "LSTM" else (N_ENCODED, 1)
    assert A_tr.ndim == 3 and A_tr.shape[1:] == expected, f"unexpected neural input shape {A_tr.shape[1:]}"
    tf.keras.utils.set_random_seed(SEED)
    net = build_keras(model, N_ENCODED)
    cbs = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=KERAS_SETTINGS["patience"], restore_best_weights=True),
           tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=KERAS_SETTINGS["reduce_lr_factor"],
                                                patience=KERAS_SETTINGS["reduce_lr_patience"], min_lr=KERAS_SETTINGS["min_lr"])]
    backup_dir = DIRS["checkpoints"] / f"{proto}_{MODEL_SLUG[model]}_epoch_backup"
    prog_path = DIRS["checkpoints"] / f"{proto}_{MODEL_SLUG[model]}_progress.json"
    # Progress ledger: accumulates epoch history and wall-clock training time across interruptions.
    prog = (json.loads(prog_path.read_text()) if prog_path.is_file() and (backup_dir / "training_metadata.json").is_file()
            else {"elapsed_s": 0.0, "loss": [], "val_loss": [], "segments": 0})
    prog["segments"] += 1
    base_elapsed, seg_t0 = prog["elapsed_s"], time.perf_counter()

    class EpochProgress(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            prog["loss"].append(float(logs.get("loss", float("nan")))); prog["val_loss"].append(float(logs.get("val_loss", float("nan"))))
            prog["elapsed_s"] = base_elapsed + time.perf_counter() - seg_t0
            prog_path.write_text(json.dumps(prog))
            print(f"      [{proto}] {model} epoch {epoch + 1:>2}: loss {prog['loss'][-1]:.4f}, val_loss {prog['val_loss'][-1]:.4f}", flush=True)
    cbs.append(EpochProgress())
    # Epoch-level resume: if the kernel stops mid-training, re-running resumes from the last completed epoch.
    # (An uninterrupted run is deterministic; a resumed run may follow a slightly different early-stopping path.)
    cbs.append(tf.keras.callbacks.BackupAndRestore(backup_dir=str(backup_dir)))
    t0 = time.perf_counter()
    hist = net.fit(A_tr, ytr, validation_data=(A_va, yva), epochs=KERAS_SETTINGS["epochs"],
                   batch_size=KERAS_SETTINGS["batch_size"], callbacks=cbs, shuffle=True, verbose=0)
    fit_s = base_elapsed + time.perf_counter() - seg_t0      # total wall-clock training time across all segments
    fitted = KerasProbabilityModel(pre, net, clip, single_step=(model == "LSTM"))
    p_va = fitted.predict_proba(Xva)[:, 1]; m = compute_metrics(yva, p_va)
    epochs = len(prog["val_loss"]); best_epoch = int(np.nanargmin(prog["val_loss"])) + 1
    resumed = prog["segments"] > 1
    prog_path.unlink(missing_ok=True)
    log = [{"config_index": 0, "params": json.dumps(SEARCH_SPACES[model][0], default=jsonable), "fit_time_s": fit_s,
            "epochs_trained": epochs, "best_epoch": best_epoch, **{f"val_{k}": v for k, v in m.items()}}]
    print(f"    [{proto}] {model}: {epochs} epochs (best {best_epoch}), val macro-F1 {m['macro_f1']:.4f} ({fit_s:.0f}s)", flush=True)
    return {"params": SEARCH_SPACES[model][0], "config_index": 0, "val_metrics": m, "p_val": p_va, "fit_time_s": fit_s,
            "fitted": fitted, "extra": {"epochs_trained": epochs, "best_epoch": best_epoch, "val_loss_history": prog["val_loss"],
                                        "training_segments": prog["segments"], "resumed_after_interruption": resumed,
                                        "note": ("resumed from epoch backup; early-stopping state restarts on resume, so the path may differ "
                                                 "slightly from an uninterrupted run" if resumed else "uninterrupted training run")}}, log


def load_fitted(proto, model):
    paths = model_paths(proto, model)
    if MODEL_KIND[model] == "keras":
        return KerasProbabilityModel(joblib.load(paths["preprocessor"]), tf.keras.models.load_model(paths["model"]),
                                     LSTM_STABILISATION["input_clip_sd"] if model == "LSTM" else None, single_step=(model == "LSTM"))
    return joblib.load(paths["model"])


def search_voting(proto, model):
    members = {m: load_fitted(proto, m) for m in ENSEMBLE_MEMBERS}
    vc = assemble_voting(members)
    Xva, yva, _ = partition(proto, "validation")
    p_va = vc.predict_proba(Xva)[:, 1]
    assert np.allclose(p_va, np.mean([RESULTS[(proto, m)]["p_val"] for m in ENSEMBLE_MEMBERS], axis=0))
    m = compute_metrics(yva, p_va)
    fit_s = sum(RESULTS[(proto, mm)]["fit_time_s"] for mm in ENSEMBLE_MEMBERS)
    log = [{"config_index": 0, "params": json.dumps(SEARCH_SPACES[model][0]), "fit_time_s": fit_s, **{f"val_{k}": v for k, v in m.items()}}]
    print(f"    [{proto}] Voting Ensemble: val macro-F1 {m['macro_f1']:.4f}", flush=True)
    return {"params": SEARCH_SPACES[model][0], "config_index": 0, "val_metrics": m, "p_val": p_va, "fit_time_s": fit_s,
            "fitted": vc, "extra": {"fit_time_definition": "sum of member final-configuration fit times (no additional fitting)"}}, log


def run_unit(proto, model):
    ck = DIRS["checkpoints"] / f"{proto}_{MODEL_SLUG[model]}.joblib"
    if ck.is_file():
        res = joblib.load(ck)
        files_ok = all((DIRS["models"] / f["file"]).is_file() and sha256_of(DIRS["models"] / f["file"]) == f["sha256"]
                       for f in res["files"].values())
        if res.get("unit_version") == UNIT_VERSION and res.get("search_space") == space_for(proto, model) and files_ok:
            RESULTS[(proto, model)] = res
            TEST_EVALUATIONS[(proto, model)] = 1
            print(f"[{proto}] {model}: loaded from checkpoint (single earlier test evaluation)")
            return
    if RUN_MODE == "analysis_only":
        raise RuntimeError(f"RUN_MODE='analysis_only' but no valid checkpoint for ({proto}, {model}); run 'train_only' or 'full' first.")
    t0 = time.perf_counter()
    kind = MODEL_KIND[model]
    runner = {"sklearn": search_sklearn, "svm": search_svm, "keras": search_keras, "voting": search_voting}[kind]
    sel, log = runner(proto, model)
    res = finalize(proto, model, sel, sel["fitted"], log)
    del sel["fitted"]; gc.collect()
    joblib.dump(res, ck)
    RESULTS[(proto, model)] = res
    tm = res["test_metrics"]
    print(f"[{proto}] {model}: selected config #{res['config_index']} | test acc {tm['accuracy']:.4f}, macro-F1 {tm['macro_f1']:.4f}, "
          f"MCC {tm['mcc']:.4f}, AUROC {tm['auroc']:.4f} | unit {time.perf_counter() - t0:.0f}s", flush=True)

## 10. Five-Model Regression Check (temporal)

Before extending the benchmark, the five models of the previous package are re-run through the revised pipeline with unchanged search spaces and compared with the previously reported temporal-test values. Small differences may arise from library versions; large differences would be investigated before proceeding.

In [ ]:
PREVIOUS_FIVE = {  # full-precision temporal-test values from the previous five-model package (files(33))
    "Logistic Regression": {"accuracy": 0.965323, "balanced_accuracy": 0.8839, "macro_f1": 0.92388, "mcc": 0.857675, "auroc": 0.941479},
    "Random Forest": {"accuracy": 0.98493, "balanced_accuracy": 0.949102, "macro_f1": 0.968803, "mcc": 0.939463, "auroc": 0.969002},
    "XGBoost": {"accuracy": 0.984408, "balanced_accuracy": 0.947341, "macro_f1": 0.967673, "mcc": 0.937338, "auroc": 0.970023},
    "LightGBM": {"accuracy": 0.984669, "balanced_accuracy": 0.948367, "macro_f1": 0.968248, "mcc": 0.938393, "auroc": 0.969578},
    "MLP": {"accuracy": 0.980132, "balanced_accuracy": 0.934354, "macro_f1": 0.958405, "mcc": 0.91973, "auroc": 0.964696}}
for model in PREVIOUS_FIVE:
    run_unit("temporal", model)
reg = []
for model, prev in PREVIOUS_FIVE.items():
    for k, v in prev.items():
        new = RESULTS[("temporal", model)]["test_metrics"][k]
        reg.append({"model": model, "metric": k, "previous_files33": v, "revised": new, "abs_difference": abs(new - v)})
five_model_regression = pd.DataFrame(reg)
save_table(five_model_regression, "five_model_regression_check.csv", "audit")
MAX_REG_DIFF = five_model_regression["abs_difference"].max()
REGRESSION_OK = MAX_REG_DIFF <= 0.005
print(f"[{'PASS' if REGRESSION_OK else 'INVESTIGATE'}] five-model regression check: max |difference| = {MAX_REG_DIFF:.6f} "
      f"(previous values rounded to 6 decimals; tolerance 0.005)")

## 11. SVM (RBF) (temporal)

Kernel SVM on scaled inputs with Platt-scaled probabilities (`probability=True`), full training partition. libsvm is single-threaded, so this is the slowest classical model.

In [ ]:
run_unit("temporal", "SVM (RBF)")

## 12. 1D-CNN (temporal)

Legacy neural baseline retained for continuity (exact legacy architecture). Early stopping on the official validation partition.

In [ ]:
run_unit("temporal", "1D-CNN")

## 13. LSTM (temporal)

Legacy neural baseline retained for continuity: legacy LSTM layers (128–64) and training settings, with the documented stabilisation and input representation of Section 8 — each flow's encoded features form a single time step, so the recurrent layers act as a gated feed-forward transformation rather than modelling any sequence.

In [ ]:
run_unit("temporal", "LSTM")

## 14. Voting Ensemble (temporal)

Legacy composition: equal-weight soft voting over the selected XGBoost, LightGBM and Random Forest members of the same protocol.

In [ ]:
run_unit("temporal", "Voting Ensemble")

## 15. IID Sensitivity Analysis (secondary)

**Secondary IID sensitivity analysis.** Each model is refitted on the IID training partition with **exactly the configuration selected on temporal validation** (no second hyperparameter search). The IID protocol is a sensitivity analysis, not a second optimisation opportunity, so holding configurations fixed isolates the effect of the evaluation protocol. Neural baselines still use the IID validation partition for early stopping, and the Voting Ensemble combines the IID members. Inference time is measured once, under the primary temporal protocol, because the IID models are the same model classes applied to test partitions of the same size.

In [ ]:
for model in MODEL_ORDER:
    run_unit("iid", model)
ALL9_TEMPORAL = all(("temporal", m) in RESULTS for m in MODEL_ORDER)
ALL9_IID = all(("iid", m) in RESULTS for m in MODEL_ORDER)
print(f"[{'PASS' if ALL9_TEMPORAL else 'FAIL'}] all 9 temporal models completed; [{'PASS' if ALL9_IID else 'FAIL'}] all 9 IID models completed")


class StopExecution(Exception):
    def _render_traceback_(self):   # stops "Run All" quietly in Jupyter
        return []

if RUN_MODE == "train_only":
    print("RUN_MODE = 'train_only': all training units are checkpointed. Set RUN_MODE = 'analysis_only' (or 'full') "
          "and run all cells again to produce tables, figures and audits without retraining.")
    raise StopExecution

## 16. Validation-Only Selection Audit and Held-Out Probabilities

The selection audit re-derives each selected configuration from the stored validation search log and confirms that it is the arg-max of the predeclared validation key. Raw probabilities for all nine models are then written for validation and test partitions of both protocols; they are never copied from `generated_attack_score` or derived from `ambiguity_score`, and are not calibrated.

In [ ]:
sel_ok = True
for (proto, model), r in RESULTS.items():
    log = pd.DataFrame(r["search_log"])
    keys = [tuple(row[f"val_{k}"] for k in SELECTION_KEYS) + (-row["config_index"],) for _, row in log.iterrows()]
    best_j = max(range(len(keys)), key=lambda j: keys[j])
    sel_ok &= int(log.iloc[best_j]["config_index"]) == r["config_index"]
    sel_ok &= not any(c.startswith("test_") for c in log.columns)
VALIDATION_ONLY_OK = bool(sel_ok)
TEST_ISOLATED_OK = all(TEST_EVALUATIONS.get(k) == 1 for k in RESULTS)
print(f"[{'PASS' if VALIDATION_ONLY_OK else 'FAIL'}] validation-only model selection (selected configs re-derived from validation logs)")
print(f"[{'PASS' if TEST_ISOLATED_OK else 'FAIL'}] temporal/IID test partitions evaluated once per unit, after selection")

truth = df.set_index("flow_id")["is_attack"].astype(int)
PROB_FILES, prob_status = {}, {}
for proto in PROTOCOLS:
    for part in ("validation", "test"):
        _, y, meta = partition(proto, part)
        frames = []
        for model in MODEL_ORDER:
            p = RESULTS[(proto, model)]["p_val" if part == "validation" else "p_test"]
            frames.append(pd.DataFrame({"flow_id": meta["flow_id"], "timestamp": meta["timestamp"], "evaluation_protocol": proto,
                                        "split": part, "y_true": y, "model": model, "y_probability_attack": p,
                                        "y_pred_0p5": (p >= THRESHOLD).astype(int)}))
        frame = pd.concat(frames, ignore_index=True)
        for model in MODEL_ORDER:
            f = frame[frame.model == model]
            ok = (len(f) == len(y) and f["y_probability_attack"].between(0, 1).all() and f["y_probability_attack"].notna().all()
                  and not f["flow_id"].duplicated().any()
                  and (f["y_pred_0p5"] == (f["y_probability_attack"] >= THRESHOLD).astype(int)).all()
                  and (f["y_true"].to_numpy() == truth.loc[f["flow_id"]].to_numpy()).all())
            prob_status[(proto, part, model)] = bool(ok)
        assert len(frame) == 9 * len(y)
        path = DIRS["probabilities"] / f"{proto}_{part}_probabilities_all9.csv"
        frame.to_csv(path, index=False, float_format="%.8f")
        PROB_FILES[f"{proto}_{part}"] = {"file": path.name, "rows": len(frame), "sha256": sha256_of(path)}
for model in MODEL_ORDER:
    ok = all(prob_status[(p, s, model)] for p in PROTOCOLS for s in ("validation", "test"))
    print(f"[{'PASS' if ok else 'FAIL'}] probabilities complete and valid: {model} (temporal/IID × validation/test)")
PROBS_OK = all(prob_status.values())
assert PROBS_OK
for proto in PROTOCOLS:
    pd.read_csv(DIRS["probabilities"] / f"{proto}_test_probabilities_all9.csv")[["flow_id", "model", "y_true", "y_pred_0p5"]] \
      .to_csv(DIRS["predictions"] / f"{proto}_test_predictions_0p5_all9.csv", index=False)

## 17. Publication Tables (Table 7, Tables S1–S2) and Reference Model

Fixed logical model order; no ranking column. FPR and FNR are reported in percent. The **Step-7 reference classifier** is selected from **temporal validation** metrics only (macro-F1 → MCC → AUROC) across all nine models.

In [ ]:
def metrics_frame(proto, which="test_metrics"):
    return pd.DataFrame([{"model": m, **RESULTS[(proto, m)][which], "train_time_s": RESULTS[(proto, m)]["fit_time_s"],
                          **RESULTS[(proto, m)]["timing"]} for m in MODEL_ORDER])

temporal_test = metrics_frame("temporal"); iid_test = metrics_frame("iid")
temporal_val = pd.DataFrame([{"model": m, "config_index": RESULTS[("temporal", m)]["config_index"], **RESULTS[("temporal", m)]["val_metrics"]} for m in MODEL_ORDER])
save_table(temporal_test, "temporal_test_metrics_all9_full_precision.csv")
save_table(iid_test, "iid_test_metrics_all9_full_precision.csv")
save_table(temporal_val, "temporal_validation_metrics_all9.csv")
save_table(pd.concat([pd.DataFrame(RESULTS[(p, m)]["search_log"]).assign(protocol=p, model=m) for p in PROTOCOLS for m in MODEL_ORDER]),
           "hyperparameter_search_all9.csv")


def fmt_time(v):
    return round(v, 2) if v < 10 else round(v, 1) if v < 1000 else round(v)


def publication_table(t):
    t = t.set_index("model").loc[MODEL_ORDER]
    out = pd.DataFrame({"Model": MODEL_ORDER})
    for col, src in [("Accuracy", "accuracy"), ("Balanced Acc.", "balanced_accuracy"), ("Precision", "precision"), ("Recall", "recall"),
                     ("F1", "f1"), ("Macro-F1", "macro_f1"), ("MCC", "mcc"), ("AUROC", "auroc")]:
        out[col] = t[src].round(4).values
    out["FPR (%)"] = (100 * t["fpr"]).round(3).values
    out["FNR (%)"] = (100 * t["fnr"]).round(2).values
    out["Train Time (s)"] = [fmt_time(v) for v in t["train_time_s"]]
    out["Amortized Batch Inference (ms/sample)"] = [float(f"{v:.3g}") if pd.notna(v) else np.nan for v in t["amortized_batch_inference_ms_per_sample"]]
    return out

table7 = publication_table(temporal_test)
tableS1 = publication_table(iid_test).drop(columns=["Amortized Batch Inference (ms/sample)"])
save_table(table7, "Table7_Track1_Temporal_All9_Baselines.csv", float_format="%.4f")
save_table(tableS1, "TableS1_Track1_IID_All9_Baselines.csv", float_format="%.4f")
TABLE7_OK = len(table7) == 9 and {"Accuracy", "Balanced Acc.", "Macro-F1", "MCC"} <= set(table7.columns)

tt, it = temporal_test.set_index("model"), iid_test.set_index("model")
s2 = {"Model": MODEL_ORDER}
for key, lbl in [("accuracy", "Accuracy"), ("macro_f1", "Macro-F1"), ("mcc", "MCC"), ("auroc", "AUROC")]:
    s2[f"Temporal {lbl}"] = tt.loc[MODEL_ORDER, key].values
    s2[f"IID {lbl}"] = it.loc[MODEL_ORDER, key].values
    s2[f"{lbl} difference (IID − temporal)"] = (it.loc[MODEL_ORDER, key] - tt.loc[MODEL_ORDER, key]).values
tableS2 = pd.DataFrame(s2)
save_table(tableS2, "TableS2_Temporal_vs_IID_All9.csv")

selected_hp = pd.DataFrame([{"protocol": p, "model": m, "config_index": RESULTS[(p, m)]["config_index"],
                             "hyperparameters": json.dumps(RESULTS[(p, m)]["params"], default=jsonable),
                             "legacy_configuration": LEGACY_CONFIGS[m],
                             "validation_macro_f1": RESULTS[(p, m)]["val_metrics"]["macro_f1"],
                             "epochs_trained": RESULTS[(p, m)]["extra"].get("epochs_trained")} for p in PROTOCOLS for m in MODEL_ORDER])
save_table(selected_hp, "selected_hyperparameters_all9.csv")

ref_model = max(MODEL_ORDER, key=lambda m: (tuple(RESULTS[("temporal", m)]["val_metrics"][k] for k in ["macro_f1", "mcc", "auroc"]), -MODEL_ORDER.index(m)))
reference_selection = {"selected_model": ref_model, "selection_rule": "highest temporal-validation macro-F1; tie-break MCC, then AUROC",
                       "selection_data": "temporal validation partition only; all nine models considered; test not used",
                       "validation_metrics": {m: {k: RESULTS[("temporal", m)]["val_metrics"][k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "mcc", "auroc"]} for m in MODEL_ORDER},
                       "configuration": RESULTS[("temporal", ref_model)]["params"]}
(DIRS["manuscript_values"] / "reference_model_selection.json").write_text(json.dumps(reference_selection, indent=2, default=jsonable))
print(f"Step-7 reference classifier (temporal validation only): {ref_model}\n")
print("Table 7 — Leakage-Safe Track-1 IDS Performance under Temporal Evaluation (all nine models, threshold 0.50)")
display(table7.style.hide(axis="index").format(precision=4).format({"FPR (%)": "{:.3f}", "FNR (%)": "{:.2f}", "Train Time (s)": "{}",
        "Amortized Batch Inference (ms/sample)": "{:.3g}"}))

## 18. Bootstrap Confidence Intervals and McNemar Tests (temporal test)

Percentile 95% intervals from 1,000 class-stratified paired bootstrap resamples (seed 2024) for all nine models. **These intervals describe resampling uncertainty on this held-out test partition only; they do not capture uncertainty across external datasets or retraining.** Pairwise McNemar tests (Holm-corrected, 36 pairs) are supplementary and are not used for ranking or selection.

In [ ]:
def fast_metrics(y, p, pred):
    tp = np.sum((pred == 1) & (y == 1)); tn = np.sum((pred == 0) & (y == 0)); fp = np.sum((pred == 1) & (y == 0)); fn = np.sum((pred == 0) & (y == 1))
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    f1p = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0; f1n = 2 * tn / (2 * tn + fn + fp) if (2 * tn + fn + fp) else 0.0
    den = np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn))
    r = stats.rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return ((tp + tn) / len(y), (tpr + tnr) / 2, (f1p + f1n) / 2, (tp * tn - fp * fn) / den if den else 0.0,
            (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

_, y_te, meta_te = partition("temporal", "test")
rng = np.random.default_rng(BOOTSTRAP_SEED)
pos, neg = np.where(y_te == 1)[0], np.where(y_te == 0)[0]
boot_idx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(BOOTSTRAP_B)]
BOOT_METRICS = ["accuracy", "balanced_accuracy", "macro_f1", "mcc", "auroc"]
ci_rows = []
for m in MODEL_ORDER:
    p = RESULTS[("temporal", m)]["p_test"]; pred = (p >= THRESHOLD).astype(int)
    point = fast_metrics(y_te, p, pred)
    assert np.isclose(point[2], tt.at[m, "macro_f1"])
    S = np.array([fast_metrics(y_te[i], p[i], pred[i]) for i in boot_idx])
    for j, k in enumerate(BOOT_METRICS):
        lo, hi = np.percentile(S[:, j], [2.5, 97.5])
        ci_rows.append({"model": m, "metric": k, "estimate": point[j], "ci95_lower": lo, "ci95_upper": hi,
                        "bootstrap_resamples": BOOTSTRAP_B, "bootstrap_seed": BOOTSTRAP_SEED})
bootstrap_ci = pd.DataFrame(ci_rows)
save_table(bootstrap_ci, "temporal_bootstrap_95CI_all9.csv")
BOOTSTRAP_OK = bootstrap_ci[["ci95_lower", "ci95_upper"]].notna().all().all() and bootstrap_ci.model.nunique() == 9


def mcnemar(ea, eb):
    b, c = int(np.sum(ea & ~eb)), int(np.sum(~ea & eb))
    if b + c == 0: return b, c, float("nan"), 1.0, "none"
    if b + c < 25: return b, c, float("nan"), stats.binomtest(b, b + c, 0.5).pvalue, "exact"
    chi2 = (abs(b - c) - 1) ** 2 / (b + c); return b, c, chi2, stats.chi2.sf(chi2, 1), "chi2_cc"

errors = {m: ((RESULTS[("temporal", m)]["p_test"] >= THRESHOLD).astype(int) != y_te) for m in MODEL_ORDER}
mc = pd.DataFrame([dict(zip(["model_a", "model_b", "a_wrong_b_right", "a_right_b_wrong", "chi2", "p_value", "method"],
                            (a, b, *mcnemar(errors[a], errors[b])))) for a, b in itertools.combinations(MODEL_ORDER, 2)])
order_ = mc["p_value"].sort_values().index; k_ = len(mc)
adj = np.minimum(np.maximum.accumulate([(k_ - i) * mc.at[j, "p_value"] for i, j in enumerate(order_)]), 1.0)
mc.loc[order_, "p_holm"] = adj
save_table(mc, "McNemar_Holm_all9.csv")
print(f"Bootstrap: {BOOTSTRAP_B} resamples × 9 models; McNemar: {k_} pairs (Holm-adjusted)")
bootstrap_ci.pivot(index="model", columns="metric", values="estimate").loc[MODEL_ORDER][BOOT_METRICS].style.format(precision=4)

## 19. Attack-Family and Device-Level Error Analysis (temporal test)

Attack-family results use attack rows only (recall = TP/(TP+FN); no TN-based accuracy is computed for family subsets). Device-level results use all rows of each device. The v1.1 release has a frozen five-device taxonomy and official split, so subgroup evidence is regenerated directly rather than inherited from legacy outputs.

In [ ]:
test_ctx = meta_te.merge(df[["flow_id", "attack_type", "device_type"]], on="flow_id", how="left")
fam_rows, dev_rows = [], []
for m in MODEL_ORDER:
    p = RESULTS[("temporal", m)]["p_test"]; pred = (p >= THRESHOLD).astype(int)
    for fam, idx in test_ctx[test_ctx.attack_type != "normal"].groupby("attack_type").groups.items():
        idx = np.asarray(list(idx)); det = int(pred[idx].sum()); n = len(idx)
        fam_rows.append({"model": m, "attack_family": fam, "support": n, "detected": det, "missed": n - det,
                         "recall": det / n, "fnr": 1 - det / n, "mean_p_attack": p[idx].mean(), "median_p_attack": float(np.median(p[idx]))})
    for dev, idx in test_ctx.groupby("device_type").groups.items():
        idx = np.asarray(list(idx)); yy = y_te[idx]; mm = compute_metrics(yy, p[idx])
        dev_rows.append({"model": m, "device_type": dev, "N": len(idx), "benign": int((yy == 0).sum()), "attack": int(yy.sum()),
                         **{k: mm[k] for k in ["accuracy", "balanced_accuracy", "precision", "recall", "macro_f1", "mcc", "fpr", "fnr"]}})
attack_family_perf = pd.DataFrame(fam_rows); device_perf = pd.DataFrame(dev_rows)
save_table(attack_family_perf, "TableS_AttackFamily_Performance.csv")
save_table(device_perf, "TableS_Device_Performance.csv")
FAMILY_OK = attack_family_perf.attack_family.nunique() == 12 and attack_family_perf.model.nunique() == 9
DEVICE_OK = device_perf.device_type.nunique() == 5 and device_perf.model.nunique() == 9
fam_mean = attack_family_perf[attack_family_perf.model.isin(REPRESENTATIVE_TREES)].groupby("attack_family")["recall"].mean().sort_values()
dev_fnr = device_perf[device_perf.model.isin(REPRESENTATIVE_TREES)].groupby("device_type")["fnr"].mean().sort_values(ascending=False)
print("Mean recall of representative tree models by attack family (lowest first):"); print(fam_mean.round(4).to_string())
print("\nMean attack FNR of representative tree models by device:"); print(dev_fnr.round(4).to_string())

## 20. Strengthened Shortcut Audit (training/validation only)

**Single-feature validation AUROC for every Track-1 feature.** For each feature, a one-feature Logistic Regression (train-fitted imputation, scaling or one-hot encoding) is trained on temporal train and scored on temporal validation; the direction-adjusted AUROC is max(AUC, 1−AUC). **Categorical level audit:** training support, attack prevalence and coverage for every category level. The temporal test partition is not used.

> This audit is diagnostic and does not retrospectively alter the frozen Track-1 feature definition.

In [ ]:
Xtr_a, ytr_a, _ = partition("temporal", "train"); Xva_a, yva_a, _ = partition("temporal", "validation")
sf_rows = []
for f in FEATURES:
    is_cat = f in CATEGORICAL
    pre = (Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
           if is_cat else Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]))
    if Xtr_a[f].nunique() <= 1:
        auc = float("nan")
    else:
        lr = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=2000))]).fit(Xtr_a[[f]], ytr_a)
        auc = roc_auc_score(yva_a, lr.predict_proba(Xva_a[[f]])[:, 1])
    sf_rows.append({"Feature": f, "Type": feature_kind(df[f]), "Validation AUROC": auc,
                    "Direction-adjusted AUROC": max(auc, 1 - auc) if not np.isnan(auc) else float("nan"),
                    "Support/levels": int(Xtr_a[f].nunique()), "Flag >= 0.95": bool(not np.isnan(auc) and max(auc, 1 - auc) >= 0.95),
                    "Note": "constant in temporal train" if Xtr_a[f].nunique() <= 1 else ""})
single_feature_auc = pd.DataFrame(sf_rows).sort_values("Direction-adjusted AUROC", ascending=False)
save_table(single_feature_auc, "single_feature_validation_auc.csv", "audit")
SF_FLAGGED = single_feature_auc.loc[single_feature_auc["Flag >= 0.95"], "Feature"].tolist()
SF_MAX = single_feature_auc.iloc[0]
SHORTCUT_OK = True
print(f"[{'PASS' if not SF_FLAGGED else 'FLAG'}] " + ("no individual Track-1 feature validation AUROC >= 0.95" if not SF_FLAGGED
      else f"features with validation AUROC >= 0.95: {SF_FLAGGED}") + f" (maximum {SF_MAX['Direction-adjusted AUROC']:.4f}, {SF_MAX['Feature']})")

cat_rows = []
for f in CATEGORICAL:
    vc = Xtr_a[f].value_counts()
    prev = pd.Series(ytr_a, index=Xtr_a.index).groupby(Xtr_a[f]).mean()
    for lvl, n in vc.items():
        pr = prev[lvl]
        cat_rows.append({"feature": f, "level": lvl, "training_support": int(n), "coverage": n / len(Xtr_a), "attack_prevalence": pr,
                         "near_deterministic": bool(pr <= 0.01 or pr >= 0.99),
                         "direction": "benign-only" if pr <= 0.01 else "attack-only" if pr >= 0.99 else ""})
categorical_audit = pd.DataFrame(cat_rows)
save_table(categorical_audit, "categorical_shortcut_audit.csv", "audit")
nd = categorical_audit[categorical_audit.near_deterministic]
ND_COVERAGE = nd.groupby("feature")["coverage"].sum().to_dict()
print("Near-deterministic category levels (training):")
print(nd[["feature", "level", "training_support", "coverage", "attack_prevalence", "direction"]].to_string(index=False))
print("Certain service-category levels show strong class association over a limited fraction of records; supplementary "
      "ablation tests are therefore used to quantify their contribution to overall model performance.")
single_feature_auc.head(10).style.format(precision=4).hide(axis="index")

## 21. Service/Port-Context Ablation (supplementary)

Predefined feature conditions, evaluated on the temporal protocol with the **already-selected primary hyperparameters** (no retuning), for the representative tree models:

* **A — Primary Track-1:** all `track1_primary_flow_only` features (results reused from the primary run; identical configuration).
* **B — minus `service_category`.**
* **C — minus service/port context:** `service_category`, `dst_port`, `port_expected_for_device`, `is_well_known_port`, `is_registered_port`, `is_dynamic_port` (where present).

The primary feature policy is not modified; this experiment only quantifies the contribution of these variables.

In [ ]:
PORT_CONTEXT = ["service_category", "dst_port", "port_expected_for_device", "is_well_known_port", "is_registered_port", "is_dynamic_port"]
CONDITIONS = {"A: primary Track-1": [], "B: minus service_category": ["service_category"],
              "C: minus service/port context": [f for f in PORT_CONTEXT if f in FEATURES]}
Xtr_t, ytr_t, _ = partition("temporal", "train")
abl_rows = []
ABL_CKPT = DIRS["checkpoints"] / "ablation.joblib"
abl_cache = joblib.load(ABL_CKPT) if ABL_CKPT.is_file() else {}
for cond, drop in CONDITIONS.items():
    keep = [f for f in FEATURES if f not in drop]
    for m in REPRESENTATIVE_TREES:
        if not drop:
            p = RESULTS[("temporal", m)]["p_test"]
        elif (cond, m, UNIT_VERSION) in abl_cache:
            p = abl_cache[(cond, m, UNIT_VERSION)]
        else:
            num_k = [f for f in NUMERIC if f in keep]; cat_k = [f for f in CATEGORICAL if f in keep]
            pre = ColumnTransformer([("num", SimpleImputer(strategy="median"), num_k),
                                     ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                                       ("o", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_k)])
            clf = build_pipeline(m, RESULTS[("temporal", m)]["params"]).named_steps["clf"]
            pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr_t[keep], ytr_t)
            Xte_k, _, _ = partition("temporal", "test")
            p = pipe.predict_proba(Xte_k[keep])[:, 1]
            abl_cache[(cond, m, UNIT_VERSION)] = p; joblib.dump(abl_cache, ABL_CKPT); del pipe
        mm = compute_metrics(y_te, p)
        abl_rows.append({"condition": cond, "removed_features": ", ".join(drop) or "none", "n_features": len(keep), "model": m,
                         **{k: mm[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "mcc", "auroc", "fpr", "fnr"]}})
ablation = pd.DataFrame(abl_rows)
base = ablation[ablation.condition.str.startswith("A")].set_index("model")
ablation["delta_macro_f1_vs_A"] = [r.macro_f1 - base.at[r.model, "macro_f1"] for r in ablation.itertuples()]
ablation["delta_fnr_vs_A"] = [r.fnr - base.at[r.model, "fnr"] for r in ablation.itertuples()]
save_table(ablation, "TableS_Service_Port_Ablation.csv")
ABLATION_OK = len(ablation) == 9
ablation.style.format(precision=4).hide(axis="index")

## 22. Single-Record Inference Timing (supplementary)

Operational timing complementary to the amortized batch measurement: after warm-up, 500 fixed temporal-test records (seed 7) are scored **one at a time** end-to-end (preprocessing + model). Median, p95 and p99 are reported. Keras models are called directly (`model(x, training=False)`) because `predict()` adds per-call overhead unrelated to the network. These values characterise the documented workstation only and do not support edge-deployment claims.

In [ ]:
SR_CKPT = DIRS["checkpoints"] / "single_record_latency.joblib"
Xte_full, _, _ = partition("temporal", "test")
sr_idx = np.random.default_rng(SINGLE_RECORD_SEED).choice(len(Xte_full), SINGLE_RECORD_N, replace=False)
if SR_CKPT.is_file() and joblib.load(SR_CKPT).get("version") == UNIT_VERSION:
    single_record = joblib.load(SR_CKPT)["table"]
else:
    sr = []
    for m in MODEL_ORDER:
        mdl = load_fitted("temporal", m)
        one = (lambda X: mdl.predict_one(X)) if MODEL_KIND[m] == "keras" else (lambda X: mdl.predict_proba(X))
        for i in sr_idx[:20]:
            one(Xte_full.iloc[[i]])
        times = []
        for i in sr_idx:
            row = Xte_full.iloc[[i]]; t0 = time.perf_counter(); one(row); times.append(1000 * (time.perf_counter() - t0))
        sr.append({"model": m, "records": SINGLE_RECORD_N, "median_ms": np.median(times), "p95_ms": np.percentile(times, 95),
                   "p99_ms": np.percentile(times, 99), "mean_ms": np.mean(times)})
        del mdl
    single_record = pd.DataFrame(sr)
    joblib.dump({"version": UNIT_VERSION, "table": single_record}, SR_CKPT)
save_table(single_record, "single_record_inference_latency.csv")
single_record.style.format(precision=3).hide(axis="index")

## 23. Revision Continuity and Legacy Comparison

The continuity audit confirms that every model family of the reviewed manuscript is present with temporal and IID results, probabilities and model binaries under the same Track-1 policy. The legacy comparison places the reviewed manuscript's Table 7 (full-feature, v1.0, 80/20 stratified diagnostic) beside the new temporal results **for traceability only**: feature policy, data version and split all differ, so no statistical comparison is made.

In [ ]:
cont = []
for m in MODEL_ORDER:
    have_t, have_i = ("temporal", m) in RESULTS, ("iid", m) in RESULTS
    probs = all(prob_status.get((p, s, m), False) for p in PROTOCOLS for s in ("validation", "test"))
    bins = all((DIRS["models"] / f["file"]).is_file() for p in PROTOCOLS for f in RESULTS[(p, m)]["files"].values()) if have_t and have_i else False
    ok = have_t and have_i and probs and bins
    cont.append({"Original model family": m, "Present in reviewed manuscript": True, "Present in revised experiment": have_t,
                 "Same Track-1 policy": True, "Temporal result available": have_t, "IID result available": have_i,
                 "Probability file available": probs, "Model binary available": bins,
                 "Legacy configuration": LEGACY_CONFIGS[m], "Status": "PASS" if ok else "FAIL"})
continuity = pd.DataFrame(cont)
save_table(continuity, "revision_continuity_audit.csv", "audit")
ALL9_RETAINED = (continuity.Status == "PASS").all()
print(f"[{'PASS' if ALL9_RETAINED else 'FAIL'}] all nine reviewed baseline families retained")

# Transcribed from the reviewed manuscript, Table 7 (percent values converted to proportions; FPR/FNR already proportions).
LEGACY_TABLE7 = {
    "Voting Ensemble": (0.9960, 1.000, 0.9731, 0.9864, 0.9957, 0.000, 0.027),
    "XGBoost": (0.9959, 0.9997, 0.9729, 0.9861, 0.9957, 0.00005, 0.027),
    "LightGBM": (0.9957, 0.9976, 0.9734, 0.9854, 0.9961, 0.00041, 0.027),
    "Random Forest": (0.9950, 1.000, 0.9664, 0.9829, 0.9946, 0.000, 0.034),
    "1D-CNN": (0.9932, 1.000, 0.9546, 0.9768, 0.9915, 0.000, 0.045),
    "MLP": (0.9924, 0.9986, 0.9505, 0.9739, 0.9906, 0.00023, 0.050),
    "LSTM": (0.9899, 0.9937, 0.9392, 0.9657, 0.9860, 0.00106, 0.061),
    "SVM (RBF)": (0.9837, 1.000, 0.8913, 0.9425, 0.9765, 0.000, 0.109),
    "Logistic Regression": (0.9781, 0.9887, 0.8636, 0.9219, 0.9803, 0.00175, 0.136)}
leg = []
for m in MODEL_ORDER:
    o = LEGACY_TABLE7[m]; n = tt.loc[m]
    leg.append({"model": m, "old_accuracy": o[0], "old_precision": o[1], "old_recall": o[2], "old_f1": o[3], "old_auroc": o[4],
                "old_fpr": o[5], "old_fnr": o[6], "new_accuracy": n.accuracy, "new_balanced_accuracy": n.balanced_accuracy,
                "new_macro_f1": n.macro_f1, "new_mcc": n.mcc, "new_auroc": n.auroc, "new_fpr": n.fpr, "new_fnr": n.fnr,
                "old_protocol": "legacy full-feature 80/20 stratified diagnostic (v1.0)",
                "new_protocol": "v1.1 leakage-safe temporal 70/15/15", "directly_comparable": False})
legacy_cmp = pd.DataFrame(leg)
save_table(legacy_cmp, "legacy_vs_v1.1_track1_comparison.csv")
continuity[["Original model family", "Temporal result available", "IID result available", "Probability file available", "Model binary available", "Status"]]

## 24. Publication Figures

### Figure 7 — Leakage-safe Track-1 benchmark overview

(a) Annotated performance matrix (fixed model order; colour scale 0–1 with power-law normalisation so that differences near 1 remain visible; printed values are exact). (b) FPR and FNR on **separate** axes because they differ by orders of magnitude. (c) Evaluation-protocol sensitivity (temporal vs IID macro-F1). (d) Efficiency–performance view: amortized batch inference time vs macro-F1, bubble area proportional to training time.

In [ ]:
fig = plt.figure(figsize=(DOUBLE_COL, 7.3))
outer = fig.add_gridspec(2, 2, hspace=0.50, wspace=0.62, width_ratios=[1.1, 0.95])
HM = ["accuracy", "balanced_accuracy", "macro_f1", "mcc", "auroc"]
HM_LBL = ["Acc.", "Bal.\nacc.", "Macro-\nF1", "MCC", "AUROC"]
ax = fig.add_subplot(outer[0, 0])
mat = tt.loc[MODEL_ORDER, HM].to_numpy()
norm = PowerNorm(gamma=4.0, vmin=0, vmax=1)
im = ax.imshow(mat, cmap=SEQ_CMAP, norm=norm, aspect="auto")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.3f}", ha="center", va="center", fontsize=7.2,
                color="white" if norm(mat[i, j]) > 0.55 else "#1A1A1A")
ax.set_xticks(range(len(HM)), HM_LBL, fontsize=7.5); ax.set_yticks(range(9), MODEL_ORDER, fontsize=7.8)
ax.xaxis.tick_top(); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_xticks(np.arange(-0.5, len(HM), 1), minor=True); ax.set_yticks(np.arange(-0.5, 9, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.0); ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02, ticks=[0.5, 0.8, 0.9, 1.0])
cb.set_label("Score (0–1, power-scaled)", fontsize=7.5); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=7, length=2)
add_panel_label(ax, "a", x=-0.62, y=1.10)

inner = outer[0, 1].subgridspec(1, 2, wspace=0.12)
ypos = np.arange(9)[::-1]
for k, (key, lbl) in enumerate([("fpr", "FPR (%)"), ("fnr", "FNR (%)")]):
    a = fig.add_subplot(inner[0, k])
    vals = 100 * tt.loc[MODEL_ORDER, key].to_numpy()
    for y_, v, m in zip(ypos, vals, MODEL_ORDER):
        a.hlines(y_, 0, v, color=MODEL_COLORS[m], lw=1.2)
        a.plot(v, y_, MODEL_MARKERS[m], color=MODEL_COLORS[m], ms=5, mec="white", mew=0.4, clip_on=False, zorder=3)
        a.text(v + vals.max() * 0.06, y_, f"{v:.2f}" if key == "fnr" else f"{v:.3f}", va="center", fontsize=6.6, color=NEUTRAL)
    a.set_xlim(0, max(vals.max() * 1.55, 0.01)); a.set_ylim(-0.6, 8.6)
    a.set_yticks(ypos, MODEL_ORDER if k == 0 else [""] * 9, fontsize=7.5); a.tick_params(axis="y", length=0)
    a.set_xlabel(lbl); ygrid(a, "x")
    if k == 0:
        add_panel_label(a, "b", x=-1.05, y=1.03)

ax = fig.add_subplot(outer[1, 0])
for y_, m in zip(ypos, MODEL_ORDER):
    tv, iv = tt.at[m, "macro_f1"], it.at[m, "macro_f1"]
    ax.plot([tv, iv], [y_ + 0.14, y_ - 0.14], color="#A0A0A0", lw=1.1, zorder=1)
    ax.plot(tv, y_ + 0.14, "o", color=MODEL_COLORS[m], ms=5.2, zorder=3, mec="white", mew=0.4)
    ax.plot(iv, y_ - 0.14, "o", mfc="white", mec=MODEL_COLORS[m], ms=5.2, mew=1.1, zorder=3)
    ax.text(1.005, y_, f"{iv - tv:+.4f}", transform=ax.get_yaxis_transform(), va="center", fontsize=6.6, color=NEUTRAL)
lo = min(tt["macro_f1"].min(), it["macro_f1"].min()); hi = max(tt["macro_f1"].max(), it["macro_f1"].max())
pad = (hi - lo) * 0.08
ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(-0.6, 8.6)
ax.set_yticks(ypos, MODEL_ORDER, fontsize=7.5); ax.tick_params(axis="y", length=0)
ax.set_xlabel("Macro-F1 (axis scaled to data range)"); ygrid(ax, "x")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=NEUTRAL, label="Temporal (primary)"),
                   Line2D([], [], marker="o", ls="", mfc="white", mec=NEUTRAL, mew=1.1, label="IID (secondary)")],
          loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=7)
ax.text(1.005, 1.0, "Δ IID−T", transform=ax.transAxes, fontsize=6.6, color=NEUTRAL, va="bottom")
add_panel_label(ax, "c", x=-0.62, y=1.10)

ax = fig.add_subplot(outer[1, 1])
xs = tt.loc[MODEL_ORDER, "amortized_batch_inference_ms_per_sample"].to_numpy(); ys = tt.loc[MODEL_ORDER, "macro_f1"].to_numpy()
tr = tt.loc[MODEL_ORDER, "train_time_s"].to_numpy()
size = lambda v: 14 + 240 * np.sqrt(v / tr.max())
ax.scatter(xs, ys, s=size(tr), c=[MODEL_COLORS[m] for m in MODEL_ORDER], alpha=0.45, edgecolors="white", linewidths=0.6, zorder=3)
ax.scatter(xs, ys, s=9, c=[MODEL_COLORS[m] for m in MODEL_ORDER], zorder=4, marker="o")
ax.set_xscale("log"); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ylo_d, yhi_d = ys.min(), ys.max(); pad = (yhi_d - ylo_d) * 0.30
xl0, xl1 = np.log10(xs.min() / 3), np.log10(xs.max() * 4); yl0, yl1 = ylo_d - pad, yhi_d + pad
ax.set_ylim(yl0, yl1); ax.set_xlim(10 ** xl0, 10 ** xl1)
fig.canvas.draw()
bb = ax.get_window_extent(); ax_w_pt, ax_h_pt = bb.width * 72 / fig.dpi, bb.height * 72 / fig.dpi
fx = (np.log10(xs) - xl0) / (xl1 - xl0); fy = (ys - yl0) / (yl1 - yl0)
r_fx = (np.sqrt(size(tr)) / 2) / ax_w_pt                      # bubble radius in axes-fraction units (x)
right = fx < 0.72                                              # label to the right of the bubble unless near the right edge
lx = np.where(right, fx + r_fx + 0.02, fx - r_fx - 0.02); ly = fy.copy()
H = 11.5 / ax_h_pt                                             # label height (~7 pt text) in axes fraction
for _ in range(500):                                           # deterministic vertical repulsion among same-side labels
    moved = False
    for i in range(9):
        for j in range(i + 1, 9):
            if right[i] == right[j] and abs(lx[i] - lx[j]) < 0.35 and abs(ly[i] - ly[j]) < H:
                d = 0.5 * (H - abs(ly[i] - ly[j])) + 1e-4
                if (ly[i], -i) >= (ly[j], -j): ly[i] += d; ly[j] -= d
                else: ly[i] -= d; ly[j] += d
                moved = True
    ly = np.clip(ly, 0.03, 0.97)
    if not moved: break
for i, m in enumerate(MODEL_ORDER):
    ax.annotate(m, (xs[i], ys[i]), xytext=(lx[i], ly[i]), textcoords="axes fraction", fontsize=6.6, color=MODEL_COLORS[m],
                ha="left" if right[i] else "right", va="center",
                arrowprops=dict(arrowstyle="-", lw=0.35, color="#A0A0A0", shrinkA=0, shrinkB=4) if abs(ly[i] - fy[i]) > H * 0.6 else None)
ref_sizes = [v for v in (1, 10, 100, 1000, 10000) if v <= tr.max() * 1.5][-3:]
ax.legend(handles=[plt.scatter([], [], s=size(v), color="#BBBBBB", alpha=0.6, edgecolors="white") for v in ref_sizes],
          labels=[f"{v:g} s" for v in ref_sizes], title="Bubble area: training time", title_fontsize=7, fontsize=6.8,
          loc="upper center", bbox_to_anchor=(0.45, -0.20), ncol=3, scatterpoints=1, columnspacing=1.8, borderpad=0.5,
          handletextpad=0.7, labelspacing=0.9)
ax.set_xlabel("Batch inference (ms/sample, log)"); ax.set_ylabel("Macro-F1 (temporal test)")
ygrid(ax, "both"); add_panel_label(ax, "d", x=-0.22, y=1.03)
CAPTION_FIG7 = (f"Figure 7. Leakage-safe Track-1 benchmark overview for all nine model families on the temporal test partition "
                f"(n = {len(y_te):,}; threshold 0.50). (a) Accuracy, balanced accuracy, macro-F1, Matthews correlation coefficient "
                "(MCC) and area under the ROC curve (AUROC); colour uses a power-law scale on 0–1 and printed values are exact. "
                "(b) False-positive rate (FPR) and false-negative rate (FNR) on separate axes. (c) Evaluation-protocol sensitivity: "
                "macro-F1 under the chronological (filled) and IID stratified (open) protocols, with the IID − temporal difference. "
                "(d) Amortized batch inference time (median end-to-end probability inference on the full test partition divided by "
                "its size) versus macro-F1; bubble area is proportional to the fitting time of the selected configuration "
                "(hyperparameter search excluded). All models use only the frozen track1_primary_flow_only features.")
save_publication_figure(fig, "Figure7_Track1_All9_Overview", CAPTION_FIG7, n_panels=4, n_models=9)
plt.show(); plt.close(fig); print(CAPTION_FIG7)

### Figure 8 — Discrimination performance on the temporal test set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, 3.6))
fig.subplots_adjust(wspace=0.28, bottom=0.30)
AP = {}
for m in MODEL_ORDER:
    p = RESULTS[("temporal", m)]["p_test"]
    fpr_c, tpr_c, _ = roc_curve(y_te, p); prec, rec, _ = precision_recall_curve(y_te, p); AP[m] = average_precision_score(y_te, p)
    axes[0].plot(fpr_c, tpr_c, color=MODEL_COLORS[m], ls=MODEL_LINESTYLES[m], lw=MODEL_LW[m])
    axes[1].plot(rec, prec, color=MODEL_COLORS[m], ls=MODEL_LINESTYLES[m], lw=MODEL_LW[m])
axes[0].plot([0, 1], [0, 1], color="#9A9A9A", lw=0.7, ls="--")
axes[0].set_xlabel("False-positive rate"); axes[0].set_ylabel("True-positive rate"); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.01)
prev = y_te.mean()
axes[1].axhline(prev, color="#9A9A9A", lw=0.7, ls="--"); axes[1].text(0.02, prev + 0.025, f"Attack prevalence {prev:.3f}", fontsize=6.8, color=NEUTRAL)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.02)
for a, l in zip(axes, "ab"):
    ygrid(a, "both"); add_panel_label(a, l, x=-0.14)
handles = [Line2D([], [], color=MODEL_COLORS[m], ls=MODEL_LINESTYLES[m], lw=MODEL_LW[m] + 0.2,
                  label=f"{m}  (AUROC {tt.at[m, 'auroc']:.3f}; AP {AP[m]:.3f})") for m in MODEL_ORDER]
fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0), fontsize=6.9, columnspacing=1.2, handlelength=2.6)
CAPTION_FIG8 = (f"Figure 8. Discrimination performance of the nine Track-1 classifiers on the held-out temporal test set (n = {len(y_te):,}). "
                "(a) Receiver-operating-characteristic (ROC) curves; the dashed diagonal is the chance reference. (b) Precision–recall "
                "curves; the dashed line is attack prevalence (no-skill precision). The legend reports AUROC and average precision (AP); "
                "each model keeps the same colour and line style in both panels.")
save_publication_figure(fig, "Figure8_Track1_ROC_PR", CAPTION_FIG8, n_panels=2, n_models=9)
plt.show(); plt.close(fig); print(CAPTION_FIG8)

### Figure 9 — Error and subgroup characteristics under temporal evaluation

Panels (a)–(c) use the three **predeclared** representative tree models (Random Forest, XGBoost, LightGBM); panel (d) uses the Step-7 reference classifier selected on temporal validation only.

In [ ]:
FAM_LBL = {"botnet_communication": "Botnet comm.", "port_scanning": "Port scanning", "dns_tunneling": "DNS tunnelling",
           "remote_administration": "Remote admin.", "device_hijacking": "Device hijacking", "network_mapping": "Network mapping",
           "protocol_abuse": "Protocol abuse", "bulk_data_transfer": "Bulk data transfer", "service_enumeration": "Service enum.",
           "credential_harvesting": "Credential harv.", "configuration_tampering": "Config. tampering", "firmware_exploitation": "Firmware exploit."}
fig = plt.figure(figsize=(DOUBLE_COL, 7.0))
outer = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.55, width_ratios=[1.05, 1.0])

ax = fig.add_subplot(outer[0, 0])
fam = attack_family_perf[attack_family_perf.model.isin(REPRESENTATIVE_TREES)]
fam_order = fam.groupby("attack_family")["recall"].mean().sort_values().index.tolist()
M = fam.pivot(index="attack_family", columns="model", values="recall").loc[fam_order, REPRESENTATIVE_TREES].to_numpy()
sup = fam[fam.model == REPRESENTATIVE_TREES[0]].set_index("attack_family").loc[fam_order, "support"]
nrm = PowerNorm(gamma=3.0, vmin=0, vmax=1)
im = ax.imshow(M, cmap=SEQ_CMAP, norm=nrm, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=6.8, color="white" if nrm(M[i, j]) > 0.55 else "#1A1A1A")
ax.set_yticks(range(len(fam_order)), [f"{FAM_LBL[f]} (n={sup[f]})" for f in fam_order], fontsize=7)
ax.set_xticks(range(3), ["RF", "XGBoost", "LightGBM"], fontsize=7.5); ax.xaxis.tick_top(); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_xlabel("Attack-family recall", fontsize=8); ax.xaxis.set_label_position("top")
cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03, ticks=[0.5, 0.8, 0.9, 1.0]); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6.8, length=2)
add_panel_label(ax, "a", x=-0.80, y=1.07)

ax = fig.add_subplot(outer[0, 1])
dv = device_perf[device_perf.model.isin(REPRESENTATIVE_TREES)]
devs = ["camera", "doorbell", "speaker", "thermostat", "tv"]
offs = {m: o for m, o in zip(REPRESENTATIVE_TREES, (0.22, 0.0, -0.22))}
for m in REPRESENTATIVE_TREES:
    d = dv[dv.model == m].set_index("device_type").loc[devs]
    yy = np.arange(len(devs))[::-1] + offs[m]
    ax.hlines(yy, 0, 100 * d["fnr"], color=MODEL_COLORS[m], lw=1.0)
    ax.plot(100 * d["fnr"], yy, MODEL_MARKERS[m], color=MODEL_COLORS[m], ms=4.8, mec="white", mew=0.4, label=m, clip_on=False, zorder=3)
nat = device_perf[device_perf.model == REPRESENTATIVE_TREES[0]].set_index("device_type").loc[devs, "attack"]
ax.set_yticks(np.arange(len(devs))[::-1], [f"{d.upper() if d == 'tv' else d.capitalize()} (n={nat[d]})" for d in devs], fontsize=7.5)
ax.tick_params(axis="y", length=0); ax.set_xlabel("Attack FNR by device (%)"); ax.set_xlim(0, None); ygrid(ax, "x")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=7, handletextpad=0.2, columnspacing=0.9)
add_panel_label(ax, "b", x=-0.40, y=1.07)

inner = outer[1, 0].subgridspec(1, 3, wspace=0.10)
cm_cmap = LinearSegmentedColormap.from_list("cm", ["#F7F9FB", "#9EB9D0", "#2B5C8A"])
for k, m in enumerate(REPRESENTATIVE_TREES):
    a = fig.add_subplot(inner[0, k]); r = tt.loc[m]
    cm = np.array([[r.tn, r.fp], [r.fn, r.tp]], dtype=float); rn = cm / cm.sum(axis=1, keepdims=True)
    a.imshow(rn, cmap=cm_cmap, vmin=0, vmax=1, aspect="auto")
    for i in range(2):
        for j in range(2):
            pct = 100 * rn[i, j]; txt = f"{pct:.2f}%" if 0 < pct < 0.1 or 99.9 < pct < 100 else f"{pct:.1f}%"
            dark = rn[i, j] > 0.55
            a.text(j, i - 0.13, f"{int(cm[i, j]):,}", ha="center", va="center", fontsize=6.9, fontweight="bold", color="white" if dark else "#1A1A1A")
            a.text(j, i + 0.2, txt, ha="center", va="center", fontsize=6.1, color="white" if dark else NEUTRAL)
    a.set_xticks([0, 1], ["Benign", "Attack"], fontsize=6.8); a.set_yticks([0, 1], ["Benign", "Attack"] if k == 0 else ["", ""], fontsize=6.8)
    a.tick_params(length=0); a.set_title(m, fontsize=7.4, color=MODEL_COLORS[m], fontweight="bold", pad=3)
    for s in a.spines.values(): s.set_visible(False)
    a.set_xlabel("Predicted", fontsize=7.2)
    if k == 0:
        a.set_ylabel("True class", fontsize=7.5); add_panel_label(a, "c", x=-0.95, y=1.13)

ax = fig.add_subplot(outer[1, 1])
p_ref = RESULTS[("temporal", ref_model)]["p_test"]; pred_ref = (p_ref >= THRESHOLD).astype(int)
groups = [("Correct benign (TN)", (y_te == 0) & (pred_ref == 0), "#0072B2", "-"),
          ("False positive (FP)", (y_te == 0) & (pred_ref == 1), "#56B4E9", ":"),
          ("Correct attack (TP)", (y_te == 1) & (pred_ref == 1), "#D55E00", "-"),
          ("False negative (FN)", (y_te == 1) & (pred_ref == 0), "#882255", "--")]
for lbl, mask, col, ls in groups:
    n = int(mask.sum())
    if n:
        x = np.sort(p_ref[mask]); ax.step(x, np.arange(1, n + 1) / n, where="post", color=col, ls=ls, lw=1.3, label=f"{lbl}, n={n:,}")
    else:
        ax.plot([], [], color=col, ls=ls, label=f"{lbl}, n=0 (none)")
ax.axvline(THRESHOLD, color="#9A9A9A", lw=0.7, ls="--"); ax.text(THRESHOLD + 0.01, 0.04, "threshold 0.50", fontsize=6.6, color=NEUTRAL)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel(f"Predicted attack probability ({ref_model})"); ax.set_ylabel("Cumulative proportion")
ax.legend(loc="center right", fontsize=6.6, bbox_to_anchor=(1.0, 0.45)); ygrid(ax, "both")
add_panel_label(ax, "d", x=-0.20, y=1.03)
CAPTION_FIG9 = (f"Figure 9. Error and subgroup characteristics under the temporal protocol (temporal test partition, n = {len(y_te):,}). "
                "(a) Recall per attack family for three predeclared representative tree models (Random Forest, RF; XGBoost; LightGBM); "
                "family support in parentheses; colour uses a power-law scale on 0–1. (b) Attack false-negative rate (FNR) by device type; "
                "the number of attack flows per device is shown in parentheses. (c) Confusion matrices (counts and row-normalised "
                "percentages; attack is the positive class). (d) Empirical cumulative distributions of the predicted attack probability of "
                f"the Step-7 reference classifier ({ref_model}, selected on temporal validation only) for correct and incorrect decisions. "
                "Only independently trained Track-1 probabilities are used.")
save_publication_figure(fig, "Figure9_Track1_Error_Subgroup_Analysis", CAPTION_FIG9, n_panels=4, n_models=3)
plt.show(); plt.close(fig); print(CAPTION_FIG9)

### Supplementary figures — service/port ablation and bootstrap intervals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, 2.7))
fig.subplots_adjust(wspace=0.55, top=0.78)
conds = list(CONDITIONS); cmark = {c: mk for c, mk in zip(conds, ("o", "s", "^"))}
cfill = {c: f for c, f in zip(conds, ("full", "none", "full"))}
for a, (key, lbl, pct) in zip(axes, [("macro_f1", "Macro-F1", False), ("mcc", "MCC", False), ("fnr", "FNR (%)", True)]):
    for j, m in enumerate(REPRESENTATIVE_TREES):
        for c_i, c in enumerate(conds):
            v = ablation[(ablation.model == m) & (ablation.condition == c)][key].iloc[0] * (100 if pct else 1)
            a.plot(j + (c_i - 1) * 0.22, v, cmark[c], color=MODEL_COLORS[m], ms=5.2, fillstyle=cfill[c], mew=1.0)
    a.set_xticks(range(3), ["RF", "XGBoost", "LightGBM"], fontsize=7.5); a.set_ylabel(lbl); ygrid(a); a.set_xlim(-0.5, 2.5)
    vals = ablation[key] * (100 if pct else 1); pad = (vals.max() - vals.min()) * 0.25 + 1e-4
    a.set_ylim(vals.min() - pad, vals.max() + pad)
for a, l in zip(axes, "abc"):
    add_panel_label(a, l, x=-0.42)
fig.legend(handles=[Line2D([], [], marker=cmark[c], ls="", color=NEUTRAL, fillstyle=cfill[c], label=c) for c in conds],
           loc="upper center", ncol=3, fontsize=7, bbox_to_anchor=(0.5, 1.0))
CAPTION_FIGSA = ("Figure S6. Service/port-context ablation on the temporal test partition for three representative tree models with the "
                 "selected primary hyperparameters (no retuning): (a) macro-F1, (b) MCC, (c) FNR. Condition A uses all primary Track-1 "
                 "features; B removes service_category; C removes service_category, dst_port, port_expected_for_device and the port-range "
                 "indicators. Vertical axes are scaled to the data range.")
save_publication_figure(fig, "FigureS_Service_Port_Ablation", CAPTION_FIGSA, n_panels=3, n_models=3)
plt.show(); plt.close(fig); print(CAPTION_FIGSA)

fig, axes = plt.subplots(1, 4, figsize=(DOUBLE_COL, 3.0), sharey=True)
fig.subplots_adjust(wspace=0.12)
ci = bootstrap_ci.set_index(["model", "metric"])
for a, (key, lbl) in zip(axes, [("accuracy", "Accuracy"), ("balanced_accuracy", "Balanced acc."), ("macro_f1", "Macro-F1"), ("mcc", "MCC")]):
    for y_, m in zip(ypos, MODEL_ORDER):
        e, lo_, hi_ = ci.loc[(m, key), ["estimate", "ci95_lower", "ci95_upper"]]
        a.errorbar(e, y_, xerr=[[e - lo_], [hi_ - e]], fmt=MODEL_MARKERS[m], color=MODEL_COLORS[m], ms=4.2, elinewidth=0.9, capsize=1.5, mec="white", mew=0.3)
    a.set_xlabel(lbl); ygrid(a, "x"); a.tick_params(axis="y", length=0)
    a.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3))
axes[0].set_yticks(ypos, MODEL_ORDER, fontsize=7.5)
CAPTION_FIGSB = (f"Figure S7. Point estimates and 95% bootstrap intervals ({BOOTSTRAP_B:,} class-stratified paired resamples of the temporal "
                 "test partition) for accuracy, balanced accuracy, macro-F1 and MCC. Intervals describe test-set resampling uncertainty only. "
                 "Horizontal axes are scaled to each metric's range.")
save_publication_figure(fig, "FigureS_Bootstrap_CI_All9", CAPTION_FIGSB, n_panels=4, n_models=9)
plt.show(); plt.close(fig); print(CAPTION_FIGSB)
FIGURES_MADE = True

## 25. Manuscript Evidence Files (generated from results)

All statements below are assembled from computed values after execution; nothing is pre-written about which model performs best.

In [ ]:
rng_ = lambda k: (tt[k].min(), tt[k].idxmin(), tt[k].max(), tt[k].idxmax())
acc_r, mf1_r, mcc_r, auc_r = rng_("accuracy"), rng_("macro_f1"), rng_("mcc"), rng_("auroc")
gap = tableS2.set_index("Model")["Macro-F1 difference (IID − temporal)"]
fam3 = attack_family_perf[attack_family_perf.model.isin(REPRESENTATIVE_TREES)].groupby("attack_family")["recall"].mean().sort_values()
fam_all = attack_family_perf.groupby("attack_family")["missed"].sum().sort_values(ascending=False)
dev3 = device_perf[device_perf.model.isin(REPRESENTATIVE_TREES)].groupby("device_type")["fnr"].mean().sort_values(ascending=False)
abl = ablation.set_index(["condition", "model"])
dC = ablation[ablation.condition.str.startswith("C")].set_index("model")["delta_macro_f1_vs_A"]
dB = ablation[ablation.condition.str.startswith("B")].set_index("model")["delta_macro_f1_vs_A"]
old_new = legacy_cmp.set_index("model")
acc_drop = (old_new["old_accuracy"] - old_new["new_accuracy"])
findings = [
    f"1. Accuracy on the temporal test partition ranges from {acc_r[0]:.4f} ({acc_r[1]}) to {acc_r[2]:.4f} ({acc_r[3]}) across the nine models.",
    f"2. Macro-F1 ranges from {mf1_r[0]:.4f} ({mf1_r[1]}) to {mf1_r[2]:.4f} ({mf1_r[3]}).",
    f"3. MCC ranges from {mcc_r[0]:.4f} ({mcc_r[1]}) to {mcc_r[2]:.4f} ({mcc_r[3]}); AUROC from {auc_r[0]:.4f} to {auc_r[2]:.4f}.",
    f"4. IID − temporal macro-F1 differences range from {gap.min():+.4f} to {gap.max():+.4f} (evaluation-protocol sensitivity). Class "
    "prevalence remains broadly stable across the generated period, while the temporal protocol enforces chronological separation; the "
    "generator was not designed to impose a systematic concept or covariate shift, so the temporal experiment evaluates chronological "
    "holdout performance rather than robustness to deliberately induced temporal drift.",
    f"5. Lowest mean recall among the representative tree models: {', '.join(f'{f} ({v:.3f})' for f, v in fam3.head(3).items())}. "
    f"Most missed attacks summed over all nine models: {', '.join(f'{f} ({v:,})' for f, v in fam_all.head(3).items())}.",
    f"6. Highest mean attack FNR among the representative tree models by device: {', '.join(f'{d} ({100*v:.2f}%)' for d, v in dev3.head(3).items())}; "
    f"lowest: {dev3.index[-1]} ({100*dev3.iloc[-1]:.2f}%).",
    f"7. Removing service_category changes macro-F1 by {dB.min():+.4f} to {dB.max():+.4f}; removing all service/port context changes it by "
    f"{dC.min():+.4f} to {dC.max():+.4f} (representative tree models, no retuning). "
    + ("High Track-1 performance is therefore not solely attributable to explicit service/port context."
       if dC.min() > -0.02 else "Service/port context is therefore an important contributor to separability."),
    f"8. The highest single-feature validation AUROC is {SF_MAX['Direction-adjusted AUROC']:.4f} ({SF_MAX['Feature']}); "
    + ("no individual Track-1 feature reaches 0.95." if not SF_FLAGGED else f"features reaching 0.95: {', '.join(SF_FLAGGED)}."),
    f"9. Step-7 reference classifier (temporal validation only; macro-F1 → MCC → AUROC): {ref_model} "
    f"(validation macro-F1 {RESULTS[('temporal', ref_model)]['val_metrics']['macro_f1']:.4f}).",
    f"10. Relative to the legacy full-feature diagnostic Table 7, accuracy is lower by {acc_drop.min():.4f} to {acc_drop.max():.4f} across models. "
    "Performance decreases modestly under the more conservative evaluation design while remaining strong across independent classifiers; "
    "the two settings differ in data version, feature policy and split, so the values are not directly comparable.",
]
(DIRS["manuscript_values"] / "key_findings.md").write_text("# Key findings (generated)\n\n" + "\n\n".join(findings), encoding="utf-8")

sec5 = f'''# Section 5 revision evidence (generated from results)

The original submission evaluated nine model families under an 80/20 stratified hold-out and broader diagnostic feature configuration. The revision retains all nine model families but reevaluates them under the frozen HITL-IoT v1.1 `track1_primary_flow_only` policy ({len(FEATURES)} source features, {N_ENCODED} encoded inputs) and a chronological 70/15/15 train/validation/test protocol. Generated construction signals, oversight metadata, identifiers and target labels are excluded from all model inputs. The revised scores therefore provide a more conservative leakage-safe reference benchmark while preserving continuity with the original reviewed experiment.

On the temporal test partition (n = {len(y_te):,}), accuracy ranged from {acc_r[0]:.4f} ({acc_r[1]}) to {acc_r[2]:.4f} ({acc_r[3]}), balanced accuracy from {tt["balanced_accuracy"].min():.4f} to {tt["balanced_accuracy"].max():.4f}, macro-F1 from {mf1_r[0]:.4f} to {mf1_r[2]:.4f}, MCC from {mcc_r[0]:.4f} to {mcc_r[2]:.4f} and AUROC from {auc_r[0]:.4f} to {auc_r[2]:.4f}. False-positive rates ranged from {100*tt["fpr"].min():.3f}% to {100*tt["fpr"].max():.3f}% and false-negative rates from {100*tt["fnr"].min():.2f}% to {100*tt["fnr"].max():.2f}%. Under the secondary IID protocol, macro-F1 differed from the temporal value by {gap.min():+.4f} to {gap.max():+.4f}.

{chr(10).join(findings)}
'''
(DIRS["manuscript_values"] / "section5_revision_evidence.md").write_text(sec5, encoding="utf-8")

rev3 = f'''# Reviewer 3 — Track-1 experimental evidence (factual; not the rebuttal letter)

- **All nine reviewed model families retained:** {", ".join(MODEL_ORDER)} — continuity audit status {"PASS" if ALL9_RETAINED else "FAIL"} for every family.
- **Construction signals excluded:** generated_attack_score, generated_prediction, ambiguity_score, all oversight fields, labels and identifiers are excluded by a programmatic leakage audit (all checks PASS).
- **Chronological evaluation:** temporal 70/15/15 with strict timestamp boundaries is the primary protocol; hyperparameters and the Step-7 reference model were selected on validation only; each test partition was evaluated once per model after selection.
- **IID sensitivity:** all nine models re-run on the IID 70/15/15 split; macro-F1 differences (IID − temporal) {gap.min():+.4f} to {gap.max():+.4f}.
- **Independent probabilities:** raw held-out probabilities for all nine models (validation and test, both protocols) are released; none is derived from generator scores.
- **Strengthened shortcut audit:** one-feature logistic models for all {len(FEATURES)} features (validation only); maximum direction-adjusted AUROC {SF_MAX["Direction-adjusted AUROC"]:.4f} ({SF_MAX["Feature"]}). Near-deterministic category levels are reported with their training coverage.
- **Service/port ablation:** removing service/port context changes macro-F1 by {dC.min():+.4f} to {dC.max():+.4f} for the representative tree models.
- **Subgroup analysis:** attack-family (12 families × 9 models) and device-level (5 devices × 9 models) performance tables are provided.
- **Replacement of legacy evidence:** the near-perfect v1.0 full-feature values (accuracy {old_new["old_accuracy"].min():.4f}–{old_new["old_accuracy"].max():.4f}) are replaced as principal evidence by leakage-safe temporal results (accuracy {acc_r[0]:.4f}–{acc_r[2]:.4f}); the legacy values are kept only in a traceability table.
- **LSTM correction (documented):** the exact legacy LSTM and a stabilised variant (gradient clipping, ±5 SD input capping) both stayed at the prevalence-baseline validation loss under the legacy 75-step sequence framing; the reported LSTM keeps the legacy layers and settings but presents each flow as a single time step (1 × 75). Decisions used train/validation evidence only; both failed learning curves are released in the audit folder.
- **Continuity note on configurations:** legacy 1D-CNN/LSTM architectures and the ensemble composition were recovered exactly; the manuscript's stated XGBoost/LightGBM settings disagree with the legacy notebook and must be corrected in the text.
'''
(DIRS["manuscript_values"] / "reviewer3_track1_evidence.md").write_text(rev3, encoding="utf-8")

sync = '''# Manuscript synchronisation checklist (do not edit the DOCX until results are frozen)

Legacy items in the reviewed manuscript that must be updated from the frozen v1.1 artefacts:

- [ ] v1.0 terminology: ml_confidence, ml_prediction, risk_score, human_reviewed, human_decision, human_confidence, decision_time, ml_human_agreement, annotator_id → v1.1 names (see variable audit of Notebook 02).
- [ ] Old device percentages (Table 2; "Mixed/Unknown") → Notebook 02 device_distribution.csv.
- [ ] Unsupported application-protocol realism (MQTT/CoAP/RTSP/DLNA emulation) → "service/port categories"; transport protocol is TCP/UDP only.
- [ ] Old configured profile accuracies (α = 0.92/0.85/0.75) → configured-profile parameters and observed balanced accuracies (Notebook 02 Table 4).
- [ ] Old file name HITL-IoT_human_annotations.csv → HITL-IoT_oversight_subset_v1.1.csv.
- [ ] Old 80/20 stratified split (102,276/25,569) → temporal 70/15/15 (primary) and IID 70/15/15 (secondary).
- [ ] Old full-feature nine-model Table 7 → Table7_Track1_Temporal_All9_Baselines.csv; legacy values only in the traceability table.
- [ ] "Inference latency (ms/sample)" → "amortized batch inference time (ms/sample)"; single-record timing in the supplement.
- [ ] Model settings paragraph: legacy notebook XGBoost max_depth=10 and LightGBM lr=0.1, n_estimators=100 contradict the manuscript text (max_depth=6; lr=0.05, n_estimators=150); replace with the selected v1.1 configurations (selected_hyperparameters_all9.csv).
- [ ] "Neural models tuned over layer sizes, dropout rates and epochs" → fixed legacy architectures with validation early stopping; state the LSTM correction (gradient clipping, ±5 SD input capping, single-time-step input) and why, citing the two failed learning curves.
- [ ] Old Figures 7–9 (diagnostic full-feature) → new Figures 7–9 of this notebook.
- [ ] Section 5.3 "subgroup analysis not reported" → attack-family and device tables now available.
- [ ] Old oversight-profile counts, disagreement and calibration numbers → Notebook 02 outputs (Step 7 will replace the oversight experiments).
- [ ] Remove any "no temporal drift" wording; use "evaluation-protocol sensitivity" and the chronological-holdout statement.
'''
(DIRS["manuscript_values"] / "manuscript_sync_checklist.md").write_text(sync, encoding="utf-8")
display(Markdown("\n\n".join(findings)))

## 26. Results JSON, Release Kit, Manifest, Checksums and Quality Control

### Public benchmark release contract

A valid public release (GitHub / dataset page) contains: this notebook with cleared outputs; the frozen `release_v1.1/` artefacts or a permanent DOI link; `benchmark_kit/` (protocol JSON, scorer, submission template, pinned requirements); the all-nine validation/test probability files; serialised models and metadata; Table 7, supplementary tables and Figures 7–9; `logs/environment.json` and `logs/benchmark_context.json` (context fingerprint); and the manifest and SHA-256 checksums. Results characterise the controlled HITL-IoT v1.1 benchmark; they do not establish operational performance on real networks or empirical human-behaviour validity.

The **release kit** turns this experiment into a reusable benchmark: a machine-readable protocol specification, a scorer that evaluates any external probability file exactly as Table 7 is computed (verified here by re-scoring all nine models), a submission template, a README and a pinned requirements file.

In [ ]:
results = {
    "feature_policy": {"name": "track1_primary_flow_only", "n_features": len(FEATURES), "n_encoded_inputs": N_ENCODED, "features": FEATURES,
                       "leakage_audit": dict(leak_checks)},
    "temporal_protocol": split_integrity[split_integrity.protocol == "temporal"].to_dict("records"),
    "iid_protocol": split_integrity[split_integrity.protocol == "iid"].to_dict("records"),
    "models": {m: {"search_space": SEARCH_SPACES[m], "legacy_configuration": LEGACY_CONFIGS[m],
                   "selected_temporal": RESULTS[("temporal", m)]["params"], "selected_iid": RESULTS[("iid", m)]["params"],
                   "files": {p: RESULTS[(p, m)]["files"] for p in PROTOCOLS}} for m in MODEL_ORDER},
    "temporal_validation_metrics": temporal_val.set_index("model").to_dict("index"),
    "temporal_test_metrics": temporal_test.set_index("model").to_dict("index"),
    "iid_test_metrics": iid_test.set_index("model").to_dict("index"),
    "temporal_bootstrap_95ci": bootstrap_ci.to_dict("records"),
    "ablation": ablation.to_dict("records"), "single_record_latency": single_record.to_dict("records"),
    "shortcut_audit": {"max_single_feature_validation_auroc": float(SF_MAX["Direction-adjusted AUROC"]), "feature": SF_MAX["Feature"],
                       "flagged_ge_0.95": SF_FLAGGED, "near_deterministic_coverage": ND_COVERAGE},
    "five_model_regression_max_abs_diff": MAX_REG_DIFF, "reference_model_selection": reference_selection,
    "timing_definitions": {"train_time_s": "fitting time of the selected final configuration; excludes hyperparameter search",
                           "amortized_batch_inference_ms_per_sample": "median end-to-end predict_proba time on the full test partition / n"},
    "probability_files": PROB_FILES, "environment": ENVIRONMENT, "benchmark_context": BENCHMARK_CONTEXT,
    "context_fingerprint": CONTEXT_FINGERPRINT, "fast_resume_mode": FAST_RESUME_MODE and HAVE_VERIFIED_CORE5_PARAMS,
    "captions": {"Figure7": CAPTION_FIG7, "Figure8": CAPTION_FIG8, "Figure9": CAPTION_FIG9, "FigureS6": CAPTION_FIGSA, "FigureS7": CAPTION_FIGSB}}
(DIRS["manuscript_values"] / "track1_benchmark_results_all9.json").write_text(json.dumps(results, indent=2, default=jsonable))

RELEASE_DIR_KIT = OUT_DIR / "benchmark_kit"; RELEASE_DIR_KIT.mkdir(exist_ok=True)
protocol_spec = {
    "benchmark": "HITL-IoT v1.1 Track-1 leakage-safe intrusion detection",
    "dataset": {"release": f"{manifest['dataset_name']} {manifest['version']}", "file": FILES["dataset"].name,
                "sha256": source_verification.set_index("File").at[FILES["dataset"].name, "Observed SHA-256"]},
    "splits": {"file": FILES["splits"].name, "primary": "temporal_split", "secondary": "iid_split",
               "partitions": {"train": "model fitting only", "validation": "hyperparameter selection / early stopping only",
                              "test": "final evaluation only; not used for any tuning"}},
    "features": {"policy_file": FILES["feature_policy"].name, "key": "track1_primary_flow_only", "list": FEATURES,
                 "prohibited_inputs": sorted(LABELS | CONSTRUCTION | OVERSIGHT | IDENTIFIERS)},
    "target": {"column": "is_attack", "positive_class": "attack (1)"},
    "decision_threshold": THRESHOLD,
    "primary_metrics": ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "macro_f1", "mcc", "auroc", "fpr", "fnr"],
    "metric_definitions": {"fpr": "FP/(FP+TN)", "fnr": "FN/(FN+TP)", "macro_f1": "unweighted mean of benign and attack F1",
                           "auroc": "area under ROC of P(attack)"},
    "submission_format": {"columns": ["flow_id", "y_probability_attack"], "rows": "exactly one per flow of the scored partition",
                          "range": "[0, 1]"},
    "reporting_rule": "report temporal-test results as primary; IID-test results only as secondary sensitivity analysis"}
(RELEASE_DIR_KIT / "track1_benchmark_protocol.json").write_text(json.dumps(protocol_spec, indent=2, default=jsonable))


def evaluate_submission(path, protocol="temporal", split="test"):
    # Score an external probability file (flow_id, y_probability_attack) exactly as Table 7 is computed.
    sub = pd.read_csv(path)
    assert {"flow_id", "y_probability_attack"} <= set(sub.columns), "submission needs flow_id and y_probability_attack"
    expected_ids = set(df.loc[df[PROTOCOLS[protocol]] == split, "flow_id"])
    assert sub["flow_id"].is_unique, "duplicate flow_id in submission"
    assert set(sub["flow_id"]) == expected_ids, "submission must contain exactly the flows of the scored partition"
    assert sub["y_probability_attack"].between(0, 1).all(), "probabilities must lie in [0, 1]"
    y = df.set_index("flow_id").loc[sub["flow_id"], "is_attack"].astype(int).to_numpy()
    return compute_metrics(y, sub["y_probability_attack"].to_numpy(float))

tmpl = df.loc[df["temporal_split"] == "test", ["flow_id"]].assign(y_probability_attack="")
tmpl.to_csv(RELEASE_DIR_KIT / "submission_template_temporal_test.csv", index=False)
all_probs = pd.read_csv(DIRS["probabilities"] / "temporal_test_probabilities_all9.csv")
scorer_ok = True
tmp = RELEASE_DIR_KIT / "_selfcheck.csv"
for m in MODEL_ORDER:
    all_probs[all_probs.model == m][["flow_id", "y_probability_attack"]].to_csv(tmp, index=False)
    got = evaluate_submission(tmp)
    scorer_ok &= all(np.isclose(got[k], tt.at[m, k], atol=1e-6) for k in ("accuracy", "macro_f1", "mcc", "auroc", "fpr", "fnr"))
tmp.unlink()
SCORER_OK = bool(scorer_ok)
print(f"[{'PASS' if SCORER_OK else 'FAIL'}] benchmark scorer reproduces Table 7 metrics for all nine models")

scorer_lines = [
    "import pandas as pd",
    "from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,",
    "                             matthews_corrcoef, precision_score, recall_score, roc_auc_score)",
    "",
    "",
    "def score(submission_csv, dataset_csv='release_v1.1/HITL-IoT_dataset_v1.1.csv',",
    "          splits_csv='release_v1.1/HITL-IoT_splits_v1.1.csv', protocol='temporal_split', split='test', threshold=0.5):",
    "    # Score a HITL-IoT v1.1 Track-1 submission (flow_id, y_probability_attack) exactly as in Table 7.",
    "    sub = pd.read_csv(submission_csv)",
    "    truth = pd.read_csv(dataset_csv, usecols=['flow_id', 'is_attack']).merge(pd.read_csv(splits_csv), on='flow_id')",
    "    truth = truth[truth[protocol] == split]",
    "    assert sub['flow_id'].is_unique and set(sub['flow_id']) == set(truth['flow_id']), 'submission must cover the partition exactly once'",
    "    assert sub['y_probability_attack'].between(0, 1).all(), 'probabilities must lie in [0, 1]'",
    "    m = truth.merge(sub, on='flow_id'); y = m['is_attack'].astype(int); p = m['y_probability_attack']",
    "    pred = (p >= threshold).astype(int)",
    "    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()",
    "    return {'accuracy': accuracy_score(y, pred), 'balanced_accuracy': balanced_accuracy_score(y, pred),",
    "            'precision': precision_score(y, pred, zero_division=0), 'recall': recall_score(y, pred, zero_division=0),",
    "            'f1': f1_score(y, pred, zero_division=0), 'macro_f1': f1_score(y, pred, average='macro', zero_division=0),",
    "            'mcc': matthews_corrcoef(y, pred), 'auroc': roc_auc_score(y, p),",
    "            'fpr': fp / (fp + tn) if fp + tn else float('nan'), 'fnr': fn / (fn + tp) if fn + tp else float('nan')}",
    "",
    "",
    "if __name__ == '__main__':",
    "    import json, sys",
    "    print(json.dumps(score(sys.argv[1]), indent=2))",
]
(RELEASE_DIR_KIT / "score_track1_submission.py").write_text("\n".join(scorer_lines) + "\n", encoding="utf-8")
(RELEASE_DIR_KIT / "requirements_track1_frozen.txt").write_text("\n".join([
    f"numpy=={np.__version__}", f"pandas=={pd.__version__}", f"scipy=={scipy.__version__}", f"scikit-learn=={sklearn.__version__}",
    f"xgboost=={xgb.__version__}", f"lightgbm=={lgb.__version__}", f"tensorflow=={tf.__version__}",
    f"matplotlib=={mpl.__version__}", f"joblib=={joblib.__version__}"]) + "\n")
readme_lines = [
    "# HITL-IoT v1.1 — Track-1 IDS benchmark package", "",
    "Generated by `03_HITL-IoT_v1.1_Track1_IDS_Benchmark_REVISED.ipynb` from the SHA-256-verified HITL-IoT v1.1 release.", "",
    "## Protocol",
    f"* Features: `track1_primary_flow_only` ({len(FEATURES)} features); labels, generator construction signals, oversight metadata and identifiers are prohibited inputs.",
    "* Primary evaluation: chronological 70/15/15 (`temporal_split`); secondary sensitivity: IID stratified 70/15/15 (`iid_split`).",
    "* Hyperparameters are selected on temporal validation only and reused unchanged for IID; each test partition is evaluated once per model; threshold 0.50.",
    "* Full specification: `benchmark_kit/track1_benchmark_protocol.json`.", "",
    "## Contents",
    "* `tables/` — Table 7 (temporal, nine models), Table S1 (IID), Table S2 (protocol sensitivity), attack-family, device, ablation, bootstrap CI, McNemar.",
    "* `probabilities/` — raw held-out P(attack) for all nine models (validation and test, both protocols); independent of generator scores.",
    "* `models/` — serialised pipelines (`.joblib`) and Keras networks (`.keras` + train-fitted preprocessor) with metadata and SHA-256.",
    "* `figures/` — Figures 7–9 and supplementary figures (PDF, SVG, 600-dpi PNG).",
    "* `audit/` — source verification, leakage audit, split integrity, shortcut audits, continuity audit, figure QC.",
    "* `benchmark_kit/` — protocol JSON, `score_track1_submission.py`, submission template, pinned requirements.", "",
    "## Evaluating a new model",
    "Produce P(attack) for every flow of the temporal test partition, save it as `flow_id,y_probability_attack`, then run",
    "`python benchmark_kit/score_track1_submission.py my_submission.csv`. Tune only on the train/validation partitions and report",
    "temporal-test results as primary.", "",
    "## Scope",
    "All data are generated by the documented HITL-IoT v1.1 generator. Results characterise this controlled benchmark and do not",
    "establish operational detection performance on real networks.",
]
(OUT_DIR / "README_track1_benchmark.md").write_text("\n".join(readme_lines) + "\n", encoding="utf-8")

REQ = {"Figure7_Track1_All9_Overview": (4, 9), "Figure8_Track1_ROC_PR": (2, 9), "Figure9_Track1_Error_Subgroup_Analysis": (4, 3),
       "FigureS_Service_Port_Ablation": (3, 3), "FigureS_Bootstrap_CI_All9": (4, 9)}
BANNED = [r"ml_confidence", r"human confidence", r"real analyst", r"no (temporal )?drift", r"real[- ]world", r"state[- ]of[- ]the[- ]art",
          r"superior", r"human[_ ]annotation", r"inference latency \(ms/sample\)"]
qc = []
for stem, (n_pan, n_mod) in REQ.items():
    info = FIGURE_REGISTRY[stem]; p = info["paths"]; img = plt.imread(p["png"])
    text = (info["caption"] + " " + info["texts"]).lower()
    row = {"figure": stem, "pdf": p["pdf"].stat().st_size > 5000 and p["pdf"].read_bytes()[:4] == b"%PDF",
           "svg": p["svg"].stat().st_size > 5000, "png_dpi_ok": img.shape[1] / info["size_in"][0] >= 290,
           "panels_expected": info["n_panels"] == n_pan, "models_expected": info["n_models"] == n_mod,
           "caption_mentions_temporal_test": "temporal test" in info["caption"].lower(),
           "terminology_ok": not any(re.search(b, text) for b in BANNED), "png_px": f"{img.shape[1]}×{img.shape[0]}"}
    row["result"] = "PASS" if all(v for k, v in row.items() if k not in ("figure", "png_px")) else "FAIL"
    qc.append(row)
figure_qc = pd.DataFrame(qc); save_table(figure_qc, "figure_quality_control.csv", "audit")
FIGURES_OK = (figure_qc.result == "PASS").all()
print(figure_qc.to_string(index=False))
print("\n(Aesthetic quality is not machine-verifiable — see the manual checklist.)")

manifest_out = {"package": OUT_DIR.name, "release": f"{manifest['dataset_name']} {manifest['version']}", "models": MODEL_ORDER,
                "primary_protocol": "temporal 70/15/15", "secondary_protocol": "IID stratified 70/15/15",
                "feature_policy": "track1_primary_flow_only", "files": {}}
skip = {"track1_benchmark_checksums.txt", "track1_benchmark_manifest.json"}
lines = []
for path in sorted(p for p in OUT_DIR.rglob("*") if p.is_file() and p.name not in skip and "checkpoints" not in p.parts):
    h = sha256_of(path); rel = path.relative_to(OUT_DIR).as_posix()
    manifest_out["files"][rel] = {"sha256": h, "bytes": path.stat().st_size}; lines.append(f"{h}  {rel}")
(OUT_DIR / "track1_benchmark_manifest.json").write_text(json.dumps(manifest_out, indent=2))
lines.append(f"{sha256_of(OUT_DIR / 'track1_benchmark_manifest.json')}  track1_benchmark_manifest.json")
(OUT_DIR / "track1_benchmark_checksums.txt").write_text("\n".join(lines) + "\n")
MANIFEST_OK = True
BINARIES_OK = all((DIRS["models"] / f["file"]).is_file() and (DIRS["models"] / f["file"]).stat().st_size > 0
                  for r in RESULTS.values() for f in r["files"].values())
print(f"\nManifest: {len(manifest_out['files'])} files hashed (checkpoints excluded; notebook hashed at delivery).")

In [ ]:
print("MANUAL VISUAL REVIEW BEFORE MANUSCRIPT SUBMISSION\n")
for item in ["All text remains readable at final journal width.", "Figure 7 panels are visually balanced.",
             "Model colours and line styles are consistent across Figures 7–9.", "Figures remain interpretable in grayscale.",
             "Legends do not obscure data.", "No misleading axis truncation; data-range axes are labelled as such.",
             "High performance is not visually exaggerated.", "Captions match the analyses.", "PDF/SVG open correctly."]:
    print(f"[ ] {item}")

checks = {
    "Frozen v1.1 source verified": RELEASE_VERIFIED, "All nine reviewed model families retained": ALL9_RETAINED,
    "Primary Track-1 feature policy verified": LEAKAGE_AUDIT_PASS, "Construction/oversight leakage excluded": LEAKAGE_AUDIT_PASS,
    "Temporal 70/15/15 split verified": TEMPORAL_SPLIT_OK, "IID 70/15/15 split verified": IID_SPLIT_OK,
    "All 9 temporal models completed": ALL9_TEMPORAL, "All 9 IID models completed": ALL9_IID,
    "Validation-only model selection": VALIDATION_ONLY_OK, "Temporal test isolated from tuning": TEST_ISOLATED_OK,
    "Accuracy retained in Table 7": "Accuracy" in table7.columns, "Balanced Accuracy added": "Balanced Acc." in table7.columns,
    "Macro-F1 and MCC added": {"Macro-F1", "MCC"} <= set(table7.columns),
    "Independent probabilities saved for all 9 models": PROBS_OK, "Single-feature shortcut audit complete": SHORTCUT_OK,
    "Service/port-context ablation complete": ABLATION_OK, "Attack-family analysis complete": FAMILY_OK,
    "Device-level analysis complete": DEVICE_OK, "Figure 7 redesigned": "Figure7_Track1_All9_Overview" in FIGURE_REGISTRY,
    "Figure 8 ROC + PR produced": "Figure8_Track1_ROC_PR" in FIGURE_REGISTRY,
    "Figure 9 error analysis produced": "Figure9_Track1_Error_Subgroup_Analysis" in FIGURE_REGISTRY,
    "Figure quality control": FIGURES_OK, "Model binaries included": BINARIES_OK, "Manifest/checksums produced": MANIFEST_OK,
    "Benchmark scorer reproduces Table 7": SCORER_OK}
bar = "=" * 78
print("\n" + bar + "\nHITL-IoT v1.1 — NINE-MODEL LEAKAGE-SAFE TRACK-1 REVISION COMPLETE\n" + bar + "\n")
for k, v in checks.items():
    print(f"{k + ':':52s}{'PASS' if v else 'FAIL'}")
print("\nPrimary evaluation:\n    Temporal 70/15/15\n\nSecondary sensitivity:\n    IID stratified 70/15/15\n")
print("Primary feature policy:\n    track1_primary_flow_only\n\nModels:\n" + "\n".join(f"    {m}" for m in MODEL_ORDER))
print(f"\nFive-model regression check vs previous package: {'PASS' if REGRESSION_OK else 'INVESTIGATE'} "
      f"(max |Δ| = {MAX_REG_DIFF:.4f}; differences can arise from library versions or hardware)")
print(f"\nStep-7 reference classifier (validation only): {ref_model}")
print("\nNEXT:\n    Freeze Track-1 results.\n    Rewrite manuscript Section 5.\n    Then conduct Step 7 selective-deferral/calibration/oversight experiments.\n" + bar)
assert all(checks.values()), [k for k, v in checks.items() if not v]